
# CS5489 Project: English-to-French Translation on a Single 16 GB GPU

**Liu, Naduo** (naduoliu2, 60777841) · **Cui, Zhaonan** (zhaonacui2, 61083251) ·
**Wang, Siming** (simwang3, 60369937)

CS5489 Machine Learning, City University of Hong Kong, 18 November 2026

All three authors contributed to the design, the experiments and the writing. Primary areas:
Liu led the data pipeline and the terminology adjudication; Cui led fine-tuning and preference
optimisation; Wang led evaluation, statistics and the routing analysis.

This notebook is the report and the code. Every experiment stage is an executable cell whose
implementation is the same file that lives under `src/`; paths are relative, so the notebook runs
on any machine that completes the setup in Section 2.1. Stage switches are off by default, which
makes a fresh run reproduce every table from the stored per-sentence scores in under two minutes;
`NB_RUN=all` turns the whole pipeline on (Section 2.4).


## 0. Shared setup

The cell below resolves project paths relative to the notebook, points the model cache at
`<repo>/hf`, and defines the reproduction switches.

| switch | effect |
| --- | --- |
| `RUN[...]` | which stage groups execute; all off by default |
| `NB_RUN=all` | environment variable the terminal launcher sets to enable every stage |
| `NB_RUN=data,sft` | enable only those groups |
| `SETUP[...]` | installation cells: `venv`, `assets`, `data` — off by default |
| `NB_SETUP=all` | run all three installation cells (Section 2) |
| `FORCE = True` | rerun a stage whose outputs already exist (`NB_FORCE=1` from a terminal) |
| `COMET_PY` | interpreter of the scoring environment, used when this kernel has no `comet` |

The notebook assumes the environments and data exist; the installation cells only run when
asked, because a machine that is already set up should not rebuild 8 GB of wheels and
weights. With every experiment switch off it recomputes the metrics and every table from
the stored per-sentence scores in about two minutes and never touches the GPU. With
`NB_RUN=all` it runs the pipeline end to end (about 10 GPU hours, Table 7). Each stage
prints a banner, a one-minute heartbeat while it runs, and its wall clock when it
finishes; a stage whose outputs already exist is skipped, so an interrupted run resumes by
re-running the notebook.

In [ ]:
import hashlib, importlib.util, io, json, os, re, subprocess, sys, threading, time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

PROJ = Path.cwd()
while not (PROJ / "src").exists() and PROJ != PROJ.parent:
    PROJ = PROJ.parent
assert (PROJ / "src").exists(), "run this notebook from inside the project tree"
ROOT = PROJ.parent
DATASET = ROOT / "Dataset"
RUNS = PROJ / "runs"

# 模型缓存：默认 <repo>/hf，环境里已有 HF_HOME 则沿用
os.environ["HF_HOME"] = str(Path(os.environ.get("HF_HOME", PROJ / "hf")).expanduser())
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")

# ---- 复现开关 --------------------------------------------------------------
# 默认全关：只重算指标与表格，约两分钟，不占 GPU。
# 终端：NB_RUN=all 全开；NB_RUN=data,sft 只开这两组；NB_FORCE=1 忽略已有产物重跑。
RUN = {"data": False, "index": False, "glossary": False, "augment": False,
       "sft": False, "dpo": False, "matrix": False, "infer": False,
       "g5": False, "score": False, "ceiling": False}
_spec = os.environ.get("NB_RUN", "").strip().lower()
if _spec in {"all", "1", "true", "yes"}:
    RUN = dict.fromkeys(RUN, True)
elif _spec:
    for _name in re.split(r"[,;\s]+", _spec):
        if _name in RUN:
            RUN[_name] = True
        elif _name:
            print(f"[warn] unknown stage group {_name!r}; known groups: {', '.join(RUN)}",
                  flush=True)
FORCE = os.environ.get("NB_FORCE", "").strip().lower() in {"1", "true", "yes"}
COMET_PY = os.environ.get("COMET_PY", "")
HEARTBEAT = float(os.environ.get("NB_HEARTBEAT", "60"))
T_START = time.time()
LOG = []

# ---- 安装开关 --------------------------------------------------------------
# 终端：NB_SETUP=all 装环境、权重与数据；NB_SETUP=venv,data 只装这两项。
SETUP = {"venv": False, "assets": False, "data": False}
_spec_setup = os.environ.get("NB_SETUP", "").strip().lower()
if _spec_setup in {"all", "1", "true", "yes"}:
    SETUP = dict.fromkeys(SETUP, True)
elif _spec_setup:
    for _name in re.split(r"[,;\s]+", _spec_setup):
        if _name in SETUP:
            SETUP[_name] = True
        elif _name:
            print(f"[warn] unknown setup group {_name!r}; known: {', '.join(SETUP)}", flush=True)


def has_comet():
    return importlib.util.find_spec("comet") is not None


def _clock(t=None):
    return time.strftime("%H:%M:%S", time.localtime(t))


def _stream(cmd):
    """子进程边跑边打印（行缓冲），返回耗时秒数；非零退出直接抛错。"""
    cmd = [str(c) for c in cmd]
    shown = [Path(c).name if i == 0 and Path(c).name.lower().startswith("python") else c
             for i, c in enumerate(cmd)]
    print("$ " + " ".join(shown[:5]) + (" …" if len(shown) > 5 else ""), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(PROJ), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                            errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line.rstrip("\n"), flush=True)
    proc.wait()
    dt = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode} after {dt:.0f}s: {' '.join(cmd[:3])}")
    return dt


def _fetch(url, dst, sha256, step=8 << 20):
    """下载到 dst 并校验 sha256；已存在且校验通过则跳过。"""
    dst = Path(dst)
    label = dst.relative_to(ROOT).as_posix() if ROOT in dst.parents else dst.name
    if dst.exists():
        have = hashlib.sha256(dst.read_bytes()).hexdigest()
        if have == sha256:
            print(f"[have] {label}  sha256 ok  ({dst.stat().st_size / 1e6:.0f} MB)", flush=True)
            return "present"
        print(f"[stale] {label}: sha256 {have[:12]} != {sha256[:12]}", flush=True)
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"# curl -L -o {label} {url}", flush=True)
    tmp = dst.with_name(dst.name + ".part")
    hasher, done, mark = hashlib.sha256(), 0, step
    with urllib.request.urlopen(url, timeout=120) as resp, tmp.open("wb") as fh:
        total = int(resp.headers.get("Content-Length") or 0)
        while True:
            buf = resp.read(1 << 20)
            if not buf:
                break
            fh.write(buf)
            hasher.update(buf)
            done += len(buf)
            if done >= mark:
                pct = f"  {done * 100 / total:.0f}%" if total else ""
                print(f"    {label}: {done / 1e6:.0f} MB{pct}", flush=True)
                mark += step
    if hasher.hexdigest() != sha256:
        tmp.unlink()
        raise RuntimeError(f"sha256 mismatch for {dst.name}")
    tmp.replace(dst)
    print(f"[ok] {label}  {dst.stat().st_size / 1e6:.0f} MB", flush=True)
    return "ok"


def _to_argv(ns):
    """把命名空间还原成命令行参数；内核里没有 comet 时用它跑同一份脚本。"""
    out = []
    for key, val in ns.items():
        if val is None or val is False:
            continue
        flag = "--" + key.replace("_", "-")
        out.append(flag) if val is True else out.extend([flag, str(val)])
    return out


def run_stage(flag, label, fn=None, outputs=(), comet=None, always=False):
    """跑一个阶段：开始横幅、心跳、计时、产物已存在则跳过；跑完记进 LOG。"""
    rec = {"stage": label, "flag": flag, "status": "skipped", "minutes": 0.0}
    LOG.append(rec)
    if not always and not RUN.get(flag, False):
        print(f"[skip] {label}: RUN['{flag}'] is False", flush=True)
        return False
    if outputs and not FORCE and all((PROJ / p).exists() for p in outputs):
        print(f"[have] {label}: {' '.join(outputs)} exists (FORCE=1 to rerun)", flush=True)
        rec["status"] = "cached"
        return False

    stop = threading.Event()

    def beat():
        t0 = time.time()
        while not stop.wait(HEARTBEAT):
            span = time.time() - t0
            shown = f"{span / 60:.0f} min" if span >= 60 else f"{span:.0f} s"
            print(f"    ... {label}: {shown} elapsed", flush=True)

    print(f"\n=== {_clock()}  {label} ===", flush=True)
    ticker = threading.Thread(target=beat, daemon=True)
    ticker.start()
    t0 = time.time()
    try:
        if comet and not has_comet():
            script, ns = comet
            if not COMET_PY:
                raise RuntimeError(
                    f"{label} needs unbabel-comet: install it in this kernel, or set "
                    "COMET_PY to the .venv-comet interpreter (see 2.1)")
            _stream([COMET_PY, "-u", script, *_to_argv(ns)])
        else:
            fn()
    except BaseException:
        rec.update(status="failed", minutes=(time.time() - t0) / 60)
        print(f"[FAIL] {label} after {(time.time() - t0) / 60:.1f} min", flush=True)
        raise
    finally:
        stop.set()
    rec.update(status="ok", minutes=(time.time() - t0) / 60)
    print(f"[done] {label} in {rec['minutes']:.1f} min ({_clock()})", flush=True)
    return True


def summary():
    """本次运行的阶段状态 + 表 3 的宏平均，作为终端输出的落点。"""
    width = max([len(r["stage"]) for r in LOG] + [5])
    print(f"\n{'stage':{width}s}  {'status':7s}  {'min':>6s}")
    print("-" * (width + 17))
    for r in LOG:
        print(f"{r['stage']:{width}s}  {r['status']:7s}  {r['minutes']:6.1f}")
    ran = sum(1 for r in LOG if r["status"] == "ok")
    print(f"\n{ran} stage(s) executed, wall clock {(time.time() - T_START) / 60:.1f} min")

    m = json.loads((PROJ / "reports" / "metrics.json").read_text(encoding="utf-8"))
    order = ["g1_zs", "g2_rag", "g3_sft", "g5_dpo_direct", "g4_sft_rag",
             "g4_sft_rag_inline", "g4_sft_rag_gated", "g5_full", "g5_no_agent",
             "g3_sft_aug"]
    print(f"\n{'run':22s}{'MTNT':>8s}{'EMEA':>8s}{'macro':>8s}")
    for rid in order:
        rec = m.get(rid)
        if not rec:
            continue
        mt = rec["by_domain"]["mtnt"]["comet_da"]
        em = rec["by_domain"]["emea"]["comet_da"]
        print(f"{rid:22s}{mt:8.4f}{em:8.4f}{(mt + em) / 2:8.4f}")

## Abstract

We build and dissect an English-to-French translation pipeline around `Qwen2.5-1.5B-Instruct`
on a single 16 GB consumer GPU, and ask which of the techniques that the proposal listed
actually buy quality. The ladder runs from zero-shot (G1) through retrieval injection (G2),
LoRA supervised fine-tuning (G3), their combination (G4), to a four-module "ultimate" system
that stacks preference optimisation, hybrid retrieval, glossary-constrained decoding, an
agentic critic/refiner loop and a quality-estimation router (G5). Every system is evaluated on
two deliberately hard registers (noisy social media, regulatory medical text) with COMET-DA,
BLEU, chrF++, terminology hit rate, latency and VRAM, and compared with paired bootstrap
tests.

Fine-tuning is the only intervention with a large effect: +0.051 macro COMET-DA over zero-shot
(0.7471 to 0.7981), significant at p < 1e-4. Preference optimisation adds a further +0.003,
below the pre-registered minimum detectable effect. Adding retrieval to the fine-tuned model
loses 0.015, the agentic review loop loses 0.024, and glossary injection is statistically
indistinguishable from zero. A ceiling experiment explains the pattern: on the same held-out
sentences a model roughly ten times larger, used zero-shot, beats our fine-tuned 1.5B model by
only 0.010 COMET-DA on medical text and 0.035 on social media. Naive data augmentation, tested
as a follow-up, degrades both domains, and the decomposition shows why. We report the negative
results with their mechanisms, the measurement protocol that makes them defensible, and a
notebook that reproduces every table.

## 1. Introduction

The course project asks for an English-to-French system, and rewards meaningful comparisons,
rigorous ablations and honest failure analysis rather than sheer model size. The natural
temptation with a small budget is to stack techniques: retrieval, preference tuning, constrained
decoding, agentic post-editing, dynamic routing. This report measures what each of them does.

The setting is one RTX 5080 with 16 GB, which rules out anything above ~3B parameters at bf16.
The base model is Qwen2.5-1.5B-Instruct, adapted with LoRA. Data comes from two registers that
stress different failure modes: MTNT (Reddit comments, typos, slang, code-switching) and EMEA
(European Medicines Agency documents, long sentences, dense terminology).

Contributions:

1. A complete G1-G5 ladder plus five leave-one-out arms and three format/pool controls,
   all run on one consumer GPU within roughly 10 GPU hours.
2. A ceiling experiment that bounds what any further investment could buy, by scoring a much
   larger model on the same sentences.
3. A negative result with a mechanism: post-SFT context injection and self-review degrade
   quality, and the degradation decomposes into identifiable causes.
4. A reproducible artefact: a notebook that regenerates every table from stored per-sentence
   scores, plus a bug log of six defects found during the runs.

## 2. Setup

**Hardware.** RTX 5080 (sm_120), 17.07 GB reported, 15.71 GB free at idle, measured
device-to-device bandwidth 771 GB/s (the marketing figure for the card is 960). PyTorch 2.11.0
with CUDA 12.8, SDPA attention, no Triton kernels.

**Two Python environments.** Training and inference run on Python 3.14.5. Scoring runs on a
separate Python 3.13.13 environment: `unbabel-comet 2.2.7` imports `functools._HashedSeq`, which
Python 3.14 removed, and its dependency pins conflict with `transformers 5.17`. Scoring is
therefore a separate process that consumes JSONL, which also keeps the COMET models out of the
generation process's VRAM.

**Data.** Table 1 lists the corpora. EMEA is delivered as 1,092,568 aligned lines. Length and
ratio filtering, exact de-duplication, and an 8-gram fingerprint pass that removes boilerplate
repeated across PDF documents leave 143,056 unique pairs, of which 36,000 are used here (27k
train, 1k dev, 2k test). Without the fingerprint pass, 26.6% of test sentences share an 8-gram
with training data, which would have inflated every retrieval result.

**Model.** LoRA with rank 16, alpha 32, all seven linear projections, 18.46M trainable parameters
(1.18% of the model). Training used `max_length=384` (covers 98.3% of MTNT training sentences
and 100% of test sentences), batch 16, cosine schedule, bf16, gradient checkpointing. Packing was
disabled after measuring that it halves throughput on this short-sentence data.

### 2.1 Environment setup

The pipeline needs two interpreters because `unbabel-comet 2.2.7` imports `functools._HashedSeq`,
removed in Python 3.14, and its pins conflict with `transformers 5.17`. Both environments are
plain `venv`. No conda, no docker. The commands below are for Windows and run from the repository
root, with `proj/` and `Dataset/` side by side. On Linux replace `.venv\Scripts\activate` with
`source .venv/bin/activate`. Cells S0.1 and S0.2 at the end of this section run exactly these
commands from inside the notebook.

```
git clone <repo> && cd <repo>
python -m venv .venv
.venv\Scripts\activate
python -m pip install -r proj/requirements-main.txt

python -m venv .venv-comet
.venv-comet\Scripts\activate
python -m pip install -r proj/requirements-comet.txt
```

`requirements-main.txt` pins `transformers 5.17.0`, `trl 1.18.0`, `peft 0.21.3`,
`datasets 5.2.0`, `sentence-transformers 6.3.0`, `faiss-cpu 1.16.0`, `sacrebleu 2.6.0`.
`requirements-comet.txt` pins `unbabel-comet 2.2.7` and `setuptools<81` (its `torchmetrics`
dependency still imports `pkg_resources`). Both files carry `--extra-index-url
https://download.pytorch.org/whl/cu128` and pin `torch==2.11.0+cu128`, so the CUDA build is not
selected by accident; anything below CUDA 12.8 fails on `sm_120` with "no kernel image is
available".

Model weights (about 8 GB: Qwen2.5-1.5B, bge-m3, COMET-DA) are fetched once into `$HF_HOME`,
which defaults to `<repo>/hf`. With the main environment active:

```
python proj/src/common/download_assets.py
```

`Unbabel/wmt22-cometkiwi-da` is a gated repository. Either accept its licence and pass a token,
or build the local checkpoint from the ungated mirror with `src/eval/prepare_qe.py`; the router
only needs the resulting `runs/qe_local/.../model.ckpt`.

### 2.2 Data placement

Three archives go into `Dataset/`, next to `proj/`. Nothing else is downloaded. Cell S0.3 fetches
the two public ones and verifies each against a SHA-256; `archive.zip` ships with the course and
is only checked.

* `archive.zip`: templated en-fr pairs, the clean sanity set (45,000 raw, 2,251 unique)
* `MTNT.1.1.tar.gz`: noisy Reddit en-fr with the official train/dev/test splits; Cell S0.3 fetches
  the same three files unpacked from a mirror of the release
* `en-fr.txt.zip`: OPUS EMEA v3, aligned en/fr text (1,092,568 lines)

`src/data/build_dataset.py` reads the archives in place, filters, de-duplicates and writes
`proj/data/processed/*.jsonl` with a SHA-256 manifest. If a file is missing the step prints a
skip line instead of failing, so the rest of the pipeline still runs.

### 2.3 Verifying the setup

The cell at the end of this section prints interpreter, GPU and package versions, checks that the
three archives are present, and reports the size of the model cache. A machine that passes it can
run every stage in this notebook.

### 2.4 Reproducing a run

Every stage reads and writes paths relative to the project, so a checkout plus the setup above is
enough. The switches sit in Section 0 of the notebook:

* everything off: metrics and every table are recomputed from the stored per-sentence scores, in
  about two minutes and without touching the GPU;
* `NB_RUN=all`: the pipeline runs end to end, from corpora and index through SFT, preference
  mining, DPO, inference and the G5 stack to COMET scoring, in about 10 GPU hours (Table 7);
* `NB_RUN=data,sft` for a subset, `NB_FORCE=1` to rerun stages whose outputs already exist.

From a terminal, `python src/report/run_notebook.py --all` executes the notebook cell by cell and
prints each stage's banner, a once-a-minute heartbeat and its wall clock as they happen.
`jupyter nbconvert --execute` writes the same notebook but buffers the output until the end.
`--setup` runs the installation cells of Section 2 first, which creates both environments,
installs the pins, and downloads the corpora and weights. A stage whose outputs exist is skipped,
so an interrupted run resumes by re-running the notebook.
Scoring needs `unbabel-comet`; if the kernel lacks it, point `COMET_PY` at the `.venv-comet`
interpreter and the cell streams that process instead.

### Cell S0.1: two Python environments

Creates `.venv` and `.venv-comet` and installs the pinned requirements, so the commands
of Section 2.1 do not have to be retyped. Set `PY_MAIN` / `PY_COMET` to the base
interpreters you want (`comet` needs Python < 3.14), or `PY_COMET` to reuse an existing
COMET environment. Off unless `SETUP['venv']` is true.

In [ ]:
# 两个解释器：命令与 §2.1 相同，只是在这里执行。
# PY_MAIN / PY_COMET 指向基础解释器（comet 需要 3.13）；COMET_PY 可复用已装好的 comet 环境。
def _imports(py, probe):
    return subprocess.run([str(py), "-c", probe], capture_output=True).returncode == 0


ENV_SPECS = [
    ("main", ".venv", "requirements-main.txt", "PY_MAIN",
     "import transformers, trl, peft, faiss"),
    ("comet", ".venv-comet", "requirements-comet.txt", "PY_COMET", "import comet"),
]
if SETUP["venv"]:
    for tag, name, req, env_var, probe in ENV_SPECS:
        target = ROOT / name
        py = target / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
        print(f"\n=== {tag}: {name} ===", flush=True)
        if not FORCE and py.exists() and _imports(py, probe):
            print(f"[have] {py.relative_to(ROOT).as_posix()} already provides {probe}", flush=True)
            continue
        if tag == "main" and not FORCE and _imports(sys.executable, probe):
            print(f"[have] the running kernel already provides `{probe}`", flush=True)
            continue
        if tag == "comet" and not FORCE and COMET_PY and _imports(COMET_PY, probe):
            print(f"[have] COMET_PY already provides `{probe}`", flush=True)
            continue
        base = os.environ.get(env_var, sys.executable)
        base_ver = subprocess.run([base, "-c", "import sys; print('%d.%d' % sys.version_info[:2])"],
                                  capture_output=True, text=True).stdout.strip()
        if tag == "comet" and tuple(int(x) for x in base_ver.split(".")) >= (3, 14):
            print(f"[stop] base python {base_ver} cannot host unbabel-comet 2.2.7; "
                  f"set {env_var} to a 3.13 interpreter and rerun", flush=True)
            continue
        if not py.exists():
            _stream([base, "-m", "venv", str(target)])
        _stream([str(py), "-m", "pip", "install", "--upgrade", "pip"])
        _stream([str(py), "-m", "pip", "install", "-r", req])
else:
    print("[skip] Cell S0.1: SETUP['venv'] is False", flush=True)


=== main: .venv ===


[have] the running kernel already provides `import transformers, trl, peft, faiss`



=== comet: .venv-comet ===


[have] COMET_PY already provides `import comet`


### Cell S0.2: model weights

Fetches Qwen2.5-1.5B, bge-m3 and the COMET checkpoints into `$HF_HOME` (about 8 GB).
The Kiwi repository is gated, so set `HF_TOKEN` if the router's local checkpoint is
not rebuilt from the mirror.

In [ ]:
# 模型权重约 8 GB，落到 $HF_HOME（默认 <repo>/hf）。gated 的 Kiwi 仓库需要 HF_TOKEN。
if SETUP["assets"]:
    py = ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    py = py if py.exists() else Path(sys.executable)
    _stream([str(py), "-u", "src/common/download_assets.py"])
else:
    print("[skip] Cell S0.2: SETUP['assets'] is False", flush=True)

$ python.exe -u src/common/download_assets.py


HF_HOME = <external>/hf


[download] qwen -> Qwen/Qwen2.5-1.5B-Instruct


[ok] qwen  snapshot 989aa798


[download] bge -> BAAI/bge-m3


[ok] bge  snapshot 5617a9f6


[download] comet_da -> Unbabel/wmt22-comet-da


[ok] comet_da  snapshot 2760a223


[download] kiwi -> Unbabel/wmt22-cometkiwi-da


[FAIL] kiwi: GatedRepoError: 401 Client Error. (Request ID: Root=1-6afe976b-0278421355fa104c60ea97c6;baf37fe5-9b6b-49ec-bf0e-287d15f0d374)


Cannot access gated repo for url https://huggingface.co/Unbabel/wmt22-cometkiwi-da/resolv


### Cell S0.3: datasets

Downloads the three archives into `Dataset/` and verifies each against a SHA-256.
The MTNT files come from a mirror of the official release; the templated
`archive.zip` is distributed with the course and is checked rather than downloaded.

In [ ]:
# 三个原始资产。MTNT 取 HF 上的官方镜像（三份 tsv 与 MTNT.1.1.tar.gz 逐字节一致），
# EMEA 取 OPUS，两个源都按 sha256 校验；archive.zip 是课程给的，需自己放进 Dataset/。
MIRRORS = [
    ("MTNT/train/train.en-fr.tsv",
     "71fc39973acf113a17151c5e5aaf21511a8135627934001c3da01ceab804fc15",
     "https://huggingface.co/datasets/FrancophonIA/MTNT/resolve/main/train/train.en-fr.tsv"),
    ("MTNT/valid/valid.en-fr.tsv",
     "b7b9c05d04730bb364bbadd770d2959849c1890355b1725fca64f22aa7c3a48a",
     "https://huggingface.co/datasets/FrancophonIA/MTNT/resolve/main/valid/valid.en-fr.tsv"),
    ("MTNT/test/test.en-fr.tsv",
     "3fc16357e89893df0082c9a3cbded4c690901e62f8dff37b066fb556c052a9c2",
     "https://huggingface.co/datasets/FrancophonIA/MTNT/resolve/main/test/test.en-fr.tsv"),
    ("en-fr.txt.zip",
     "6030ed9ce9991bedd470e9ba322830a4d5254f9c262e5abf5b7b6747736596f4",
     "https://object.pouta.csc.fi/OPUS-EMEA/v3/moses/en-fr.txt.zip"),
]
COURSE_ARCHIVE = ("archive.zip",
                  "8d86c9f4fd4e0e3297ac36cde1e2c5a77d4eed1a030e1f7c0537b09cc2544615")
if SETUP["data"]:
    for rel, sha, url in MIRRORS:
        _fetch(url, DATASET / rel, sha)
    name, sha = COURSE_ARCHIVE
    f = DATASET / name
    if not f.exists():
        print(f"[todo] Dataset/{name}: 课程的干净集（45,000 条模板句对），放进 Dataset/ 后再跑 Cell D1",
              flush=True)
    elif hashlib.sha256(f.read_bytes()).hexdigest() == sha:
        print(f"[have] Dataset/{name}  sha256 ok  ({f.stat().st_size / 1e6:.1f} MB)", flush=True)
    else:
        print(f"[warn] Dataset/{name}: 与记录的 sha256 不同，内容以文件为准", flush=True)
else:
    print("[skip] Cell S0.3: SETUP['data'] is False", flush=True)

[have] Dataset/MTNT/train/train.en-fr.tsv  sha256 ok  (11 MB)


[have] Dataset/MTNT/valid/valid.en-fr.tsv  sha256 ok  (0 MB)


[have] Dataset/MTNT/test/test.en-fr.tsv  sha256 ok  (0 MB)


[have] Dataset/en-fr.txt.zip  sha256 ok  (37 MB)


[have] Dataset/archive.zip  sha256 ok  (1.5 MB)


### Cell S1: environment check

The cell below prints interpreter, GPU and package versions, checks the three archives and the model cache.

In [ ]:
import platform

print("python   ", sys.version.split()[0], "|", platform.platform())
try:
    import torch, transformers
    print("torch    ", torch.__version__, "| cuda", torch.cuda.is_available(),
          "| transformers", transformers.__version__)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print("gpu      ", p.name, f"{p.total_memory/1e9:.1f} GB",
              "| capability", torch.cuda.get_device_capability(0))
except Exception as exc:
    print("torch    ", "unavailable:", exc)

for name in ("archive.zip", "MTNT.1.1.tar.gz", "en-fr.txt.zip"):
    f = DATASET / name
    print(f"dataset  {name:20s}", "ok" if f.exists() else "MISSING",
          f"{f.stat().st_size/1e6:.0f} MB" if f.exists() else "")

cache = Path(os.environ["HF_HOME"])
size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file()) / 1e9 if cache.exists() else 0
# 显示相对仓库的位置，避免把某台机器的绝对路径写进输出
try:
    label = cache.relative_to(PROJ).as_posix()
except ValueError:
    label = cache.name
print(f"cache    {label}  {size:.1f} GB")
if importlib.util.find_spec("comet"):
    print("comet     available in this kernel")
elif COMET_PY:
    print("comet     via COMET_PY ->", Path(COMET_PY).parent.parent.name)
else:
    print("comet     not in this kernel (install it, or set COMET_PY to .venv-comet)")

python    3.14.5 | Windows-11-10.0.26300-SP0


torch     2.11.0+cu128 | cuda True | transformers 5.17.0
gpu       NVIDIA GeForce RTX 5080 17.1 GB | capability (12, 0)
dataset  archive.zip          ok 1 MB
dataset  MTNT.1.1.tar.gz      ok 37 MB
dataset  en-fr.txt.zip        ok 37 MB
cache    hf  34.9 GB
comet     via COMET_PY -> .venv-comet


### Cell D1: build the three corpora

In [ ]:
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import random
import re
import sys
import tarfile
import zipfile
from collections import Counter
from pathlib import Path

def stage_data(args):
    ROOT = PROJ
    DATASET = ROOT.parent / "Dataset"
    PROCESSED = ROOT / "data" / "processed"
    MANIFESTS = ROOT / "data" / "manifests"
    REPORTS = ROOT / "reports"

    SEED = 42
    BLOCK = 100                                          # EMEA 文档块大小

    EMEA_PLAN = {"test": 20, "dev": 10, "memory_pool": 60, "train": 270}   # 块数
    CLEAN_PLAN = {"train": 1800, "dev": 200, "test": 251}
    MTNT_UNSEEN_POOL = 3000                              # 从 MTNT train 留出，不进 SFT

    PUNCT_ONLY = re.compile(r"^[\W\d_\.]*$")
    WS = re.compile(r"\s+")


    def sha1(text: str) -> str:
        return hashlib.sha1(text.encode("utf-8")).hexdigest()


    def norm(text: str) -> str:
        """只做空白与不可见字符修复，保留大小写与拼写噪声。"""
        return WS.sub(" ", text.replace("\u00a0", " ").replace("\t", " ")).strip()


    def wc(text: str) -> int:
        return len(text.split())


    NON_ALNUM = re.compile(r"[^a-z0-9à-ÿ' ]+")


    def canonical(text: str) -> list[str]:
        """小写 + 去标点，返回词序列，用于近重复检测（不影响落盘原文）。"""
        return NON_ALNUM.sub(" ", text.lower()).split()


    class GramIndex:
        """8-gram 指纹索引（int64 摘要），用于近重复过滤与跨集泄漏检测。

        281k 句 x ~15 gram 约 4M 个 int，占用 300-500MB 内存，32GB 机器可承受。
        """

        def __init__(self, n: int = 8):
            self.n = n
            self.seen: set[int] = set()

        def grams(self, text: str) -> set[int]:
            toks = canonical(text)
            if len(toks) < self.n:
                toks = toks + ["<pad>"] * (self.n - len(toks))
            return {int.from_bytes(hashlib.blake2b(" ".join(toks[i:i + self.n]).encode(),
                                                   digest_size=8).digest(), "big")
                    for i in range(len(toks) - self.n + 1)}

        def is_dup(self, text: str) -> bool:
            g = self.grams(text)
            return bool(g & self.seen)

        def add(self, text: str) -> None:
            self.seen |= self.grams(text)

        def add_many(self, texts) -> None:
            for t in texts:
                self.seen |= self.grams(t)


    # clean 集（archive.zip）

    def build_clean(stats: dict) -> None:
        zf = zipfile.ZipFile(DATASET / "archive.zip")
        seen: dict[str, dict] = {}
        dropped_dup = 0
        total = 0
        for name in ("train.csv", "validation.csv", "test.csv"):
            with zf.open(name) as fh:
                for row in csv.DictReader(io.TextIOWrapper(fh, encoding="utf-8")):
                    total += 1
                    src, tgt = norm(row["english"]), norm(row["french"])
                    if not src or not tgt:
                        continue
                    key = (src.lower(), tgt.lower())
                    if key in seen:
                        dropped_dup += 1
                        continue
                    seen[key] = {
                        "src": src, "tgt": tgt, "domain": "clean",
                        "category": row.get("category", ""), "difficulty": row.get("difficulty", ""),
                    }
        stats["clean"] = {"rows_read": total, "unique_pairs": len(seen), "dropped_duplicates": dropped_dup}

        items = sorted(seen.values(), key=lambda d: sha1(d["src"]))
        rng = random.Random(SEED)
        rng.shuffle(items)
        cursor = 0
        for split, n in CLEAN_PLAN.items():
            chunk = items[cursor:cursor + n]
            cursor += n
            for i, d in enumerate(chunk):
                d["id"] = f"clean_{split}_{i:05d}"
                d["split"] = split
                d["src_words"] = wc(d["src"])
            write_jsonl("clean", split, chunk)
        stats["clean"]["splits"] = CLEAN_PLAN


    # MTNT

    def read_mtnt(member: str) -> list[dict]:
        tar = DATASET / "MTNT.1.1.tar.gz"
        if tar.exists():
            with tarfile.open(tar, "r:gz") as tf:
                raw = tf.extractfile(member).read().decode("utf-8", errors="replace")
        else:
            # Notebook 的下载单元把官方 tsv 直接放到 Dataset/MTNT/<split>/ 下
            raw = (DATASET / member).read_text(encoding="utf-8", errors="replace")
        out = []
        for line in raw.splitlines():
            if not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                continue
            out.append({"comment_id": parts[0], "src": norm(parts[1]), "tgt": norm(parts[2])})
        return out


    def mtnt_ok(d: dict) -> bool:
        n = wc(d["src"])
        return bool(d["src"]) and bool(d["tgt"]) and 1 <= n <= 400


    def build_mtnt(stats: dict) -> None:
        mapping = {"train": "MTNT/train/train.en-fr.tsv",
                   "dev": "MTNT/valid/valid.en-fr.tsv",
                   "test": "MTNT/test/test.en-fr.tsv"}
        stats["mtnt"] = {}
        pools: dict[str, list[dict]] = {}
        for split, member in mapping.items():
            rows = read_mtnt(member)
            kept = [d for d in rows if mtnt_ok(d)]
            for i, d in enumerate(kept):
                d.update(id=f"mtnt_{split}_{i:05d}", domain="mtnt", split=split, src_words=wc(d["src"]))
            pools[split] = kept
            stats["mtnt"][split] = {"raw": len(rows), "kept": len(kept), "dropped": len(rows) - len(kept)}

        # train 侧去污染：任何与 dev/test 源句共享 8-gram 的训练样本都剔除。
        # dev/test 保持官方切分不动，便于与已发表结果对照。
        eval_grams = GramIndex(8)
        eval_grams.add_many([d["src"] for d in pools["dev"]] + [d["src"] for d in pools["test"]])
        before = len(pools["train"])
        pools["train"] = [d for d in pools["train"] if not eval_grams.is_dup(d["src"])]
        stats["mtnt"]["train_dropped_for_eval_overlap"] = before - len(pools["train"])
        stats["mtnt"]["train_after_decontam"] = len(pools["train"])
        print(f"  MTNT train 去污染: {before} -> {len(pools['train'])}", flush=True)

        # train 里留出 unseen 检索池：按 comment_id 分块抽，避免同帖跨池
        train = pools["train"]
        by_comment: dict[str, list[dict]] = {}
        for d in train:
            by_comment.setdefault(d["comment_id"], []).append(d)
        cids = sorted(by_comment)
        rng = random.Random(SEED)
        rng.shuffle(cids)
        unseen, sft_train = [], []
        for cid in cids:
            if len(unseen) < MTNT_UNSEEN_POOL:
                unseen.extend(by_comment[cid])
            else:
                sft_train.extend(by_comment[cid])
        stats["mtnt"]["memory_pool_unseen"] = len(unseen)
        stats["mtnt"]["train_for_sft"] = len(sft_train)

        write_jsonl("mtnt", "train", sft_train)
        write_jsonl("mtnt", "memory_pool_unseen", unseen)
        write_jsonl("mtnt", "dev", pools["dev"])
        write_jsonl("mtnt", "test", pools["test"])


    # EMEA

    def emea_keep(src: str, tgt: str) -> bool:
        if not src or not tgt or src == tgt:
            return False
        ls, lt = wc(src), wc(tgt)
        if not (4 <= ls <= 60) or lt < 3:
            return False
        if not 0.5 <= (lt / ls) <= 2.2:
            return False
        if PUNCT_ONLY.match(src) or PUNCT_ONLY.match(tgt):
            return False
        letters = sum(c.isalpha() for c in src)
        return letters >= max(8, int(0.5 * len(src)))


    def build_emea(stats: dict) -> None:
        zf = zipfile.ZipFile(DATASET / "en-fr.txt.zip")
        en = zf.open("EMEA.en-fr.en").read().decode("utf-8", errors="replace").splitlines()
        fr = zf.open("EMEA.en-fr.fr").read().decode("utf-8", errors="replace").splitlines()
        assert len(en) == len(fr), "EMEA 行不对齐"

        seen: set[tuple[str, str]] = set()
        kept: list[dict] = []
        dropped = Counter()
        for a, b in zip(en, fr):
            src, tgt = norm(a), norm(b)
            if not src or not tgt:
                dropped["empty"] += 1
                continue
            if not emea_keep(src, tgt):
                dropped["filter"] += 1
                continue
            key = (src.lower(), tgt.lower())
            if key in seen:
                dropped["duplicate"] += 1
                continue
            seen.add(key)
            kept.append({"src": src, "tgt": tgt})

        stats["emea"] = {"lines": len(en), "kept_unique": len(kept), "dropped": dict(dropped)}

        # 近重复过滤：EMEA 来自 PDF，跨文档的样板句（免责声明、章节标题）大量重复。
        # 只做精确去重时，test 与 train 的 8-gram 重叠率高达 26.6%（实测），
        # 因此这里对全量候选做一次 8-gram 指纹过滤，再切分。
        idx = GramIndex(8)
        dedup: list[dict] = []
        near_dup = 0
        for d in kept:
            if idx.is_dup(d["src"]):
                near_dup += 1
                continue
            idx.add(d["src"])
            dedup.append(d)
        stats["emea"]["near_dup_dropped"] = near_dup
        stats["emea"]["after_near_dup"] = len(dedup)
        print(f"  EMEA 近重复过滤: {len(kept)} -> {len(dedup)} (drop {near_dup})", flush=True)
        kept = dedup

        rng = random.Random(SEED + 1)
        n_blocks = len(kept) // BLOCK
        block_ids = list(range(n_blocks))
        rng.shuffle(block_ids)
        cursor = 0
        for split, n_blocks_split in EMEA_PLAN.items():
            take = block_ids[cursor:cursor + n_blocks_split]
            cursor += n_blocks_split
            rows = []
            for bi in sorted(take):
                rows.extend(kept[bi * BLOCK:(bi + 1) * BLOCK])
            for i, d in enumerate(rows):
                d.update(id=f"emea_{split}_{i:05d}", domain="emea", split=split, src_words=wc(d["src"]))
            write_jsonl("emea", split, rows)
            stats["emea"][split] = len(rows)



    def write_jsonl(domain: str, split: str, rows: list[dict]) -> Path:
        PROCESSED.mkdir(parents=True, exist_ok=True)
        MANIFESTS.mkdir(parents=True, exist_ok=True)
        path = PROCESSED / f"{domain}.{split}.jsonl"
        h = hashlib.sha256()
        with path.open("w", encoding="utf-8", newline="\n") as fh:
            for d in rows:
                line = json.dumps(d, ensure_ascii=False, sort_keys=True)
                fh.write(line + "\n")
                h.update(line.encode("utf-8"))
        manifest = {"file": str(path.relative_to(ROOT)), "rows": len(rows), "sha256": h.hexdigest(),
                    "domain": domain, "split": split, "seed": SEED}
        (MANIFESTS / f"{domain}.{split}.json").write_text(
            json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"  wrote {path.name}: {len(rows)} rows  sha256={h.hexdigest()[:12]}", flush=True)
        return path


    def load_jsonl(domain: str, split: str) -> list[dict]:
        path = PROCESSED / f"{domain}.{split}.jsonl"
        with path.open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def ngrams(text: str, n: int = 8) -> set[str]:
        toks = text.lower().split()
        return {" ".join(toks[i:i + n]) for i in range(max(0, len(toks) - n + 1))}


    def leakage_check(stats: dict) -> None:
        """每个域的 test/dev 与 train+memory_pool 做句级 hash 与 8-gram 重叠检查。"""
        leaks = {}
        for domain in ("clean", "mtnt", "emea"):
            train_src = set()
            gram = set()
            for split in ("train", "memory_pool", "memory_pool_unseen"):
                path = PROCESSED / f"{domain}.{split}.jsonl"
                if not path.exists():
                    continue
                for d in load_jsonl(domain, split):
                    train_src.add(d["src"].strip().lower())
                    gram |= ngrams(d["src"])
            for split in ("dev", "test"):
                path = PROCESSED / f"{domain}.{split}.jsonl"
                if not path.exists():
                    continue
                rows = load_jsonl(domain, split)
                exact = sum(1 for d in rows if d["src"].strip().lower() in train_src)
                g = sum(1 for d in rows if ngrams(d["src"]) & gram)
                leaks[f"{domain}.{split}"] = {
                    "rows": len(rows), "exact_src_overlap": exact, "8gram_overlap": g,
                    "8gram_rate": round(g / max(1, len(rows)), 4),
                }
                print(f"  leakage {domain}.{split}: exact={exact} 8gram={g}", flush=True)
        stats["leakage"] = leaks


    def length_stats(stats: dict) -> None:
        out = {}
        # 只统计本脚本产出的三个域；processed 目录里还有后续步骤生成的文件（chat、DPO 对、增强集）
        produced = [p for p in sorted(PROCESSED.glob("*.jsonl"))
                    if p.name.split(".")[0] in {"clean", "mtnt", "emea"}]
        for path in produced:
            rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
            ws = sorted(d["src_words"] for d in rows)
            wt = sorted(wc(d["tgt"]) for d in rows)
            if not ws:
                continue
            q = lambda v, p: v[min(len(v) - 1, int(len(v) * p / 100))]
            out[path.stem] = {
                "rows": len(rows),
                "src_words": {"p50": q(ws, 50), "p90": q(ws, 90), "p95": q(ws, 95),
                              "p99": q(ws, 99), "max": ws[-1]},
                "tgt_words": {"p50": q(wt, 50), "p95": q(wt, 95), "max": wt[-1]},
                "est_src_tokens_p99": int(q(ws, 99) * 1.35),
            }
        stats["length"] = out


    def main(args):

        stats: dict = {"seed": SEED, "emea_block": BLOCK}
        print("[1/5] clean (archive.zip)", flush=True)
        build_clean(stats)
        print("[2/5] MTNT", flush=True)
        build_mtnt(stats)
        if not args.skip_emea:
            print("[3/5] EMEA (1.09M 行，约 2-3 分钟)", flush=True)
            build_emea(stats)
        print("[4/5] 泄漏检查", flush=True)
        leakage_check(stats)
        print("[5/5] 长度统计", flush=True)
        length_stats(stats)

        REPORTS.mkdir(parents=True, exist_ok=True)
        (REPORTS / "data_stats.json").write_text(
            json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
        print("\nDONE -> reports/data_stats.json", flush=True)

    main(args)

run_stage('data', 'Cell D1: build the three corpora',
          fn=lambda: stage_data(SimpleNamespace(**{'skip_emea': None})),
          outputs=('data/processed/emea.test.jsonl', 'data/processed/mtnt.test.jsonl'))

[skip] Cell D1: build the three corpora: RUN['data'] is False


False

### Cell D2: retrieval index

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import pickle
import sys
import time
from pathlib import Path


import os

def stage_index(args):
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    import numpy as np
    import torch

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    INDEX = ROOT / "data" / "index"
    ENCODER = "BAAI/bge-m3"

    POOLS = {
        "seen": ["mtnt.train.jsonl", "emea.train.jsonl", "clean.train.jsonl"],
        "unseen": ["mtnt.memory_pool_unseen.jsonl", "emea.memory_pool.jsonl"],
    }


    def load_pool(names: list[str], limit: int | None) -> list[dict]:
        rows: list[dict] = []
        for name in names:
            path = DATA / name
            if not path.exists():
                print(f"  [skip] {name} 不存在", flush=True)
                continue
            with path.open(encoding="utf-8") as fh:
                for line in fh:
                    if line.strip():
                        rows.append(json.loads(line))
        if limit:
            rng = np.random.default_rng(42)
            idx = rng.choice(len(rows), size=min(limit, len(rows)), replace=False)
            rows = [rows[i] for i in sorted(idx)]
        return rows


    def encode(texts: list[str], batch: int = 128) -> np.ndarray:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(ENCODER, device="cuda", model_kwargs={"torch_dtype": torch.float16})
        t0 = time.perf_counter()
        emb = model.encode(texts, batch_size=batch, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=True)
        print(f"  encoded {len(texts)} 句，用时 {time.perf_counter()-t0:.1f}s，"
              f"峰值显存 {torch.cuda.max_memory_allocated()/1e9:.2f}GB", flush=True)
        del model
        torch.cuda.empty_cache()
        return emb.astype(np.float32)


    def main(args):

        import faiss
        from rank_bm25 import BM25Okapi

        INDEX.mkdir(parents=True, exist_ok=True)
        rows = load_pool(POOLS[args.pool], args.limit)
        print(f"[{args.pool}] 句对 {len(rows)}", flush=True)

        srcs = [r["src"] for r in rows]
        emb = encode(srcs)

        index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb)
        faiss_path = INDEX / f"{args.pool}.faiss"
        faiss.write_index(index, str(faiss_path))

        tok = [s.lower().split() for s in srcs]
        bm25 = BM25Okapi(tok)
        bm25_path = INDEX / f"{args.pool}.bm25.pkl"
        with bm25_path.open("wb") as fh:
            pickle.dump(bm25, fh)

        meta = INDEX / f"{args.pool}.meta.jsonl"
        with meta.open("w", encoding="utf-8", newline="\n") as fh:
            for r in rows:
                fh.write(json.dumps({"id": r["id"], "domain": r["domain"], "src": r["src"],
                                     "tgt": r["tgt"]}, ensure_ascii=False) + "\n")

        print(f"  -> {faiss_path.name} ({faiss_path.stat().st_size/1e6:.1f}MB), "
              f"{bm25_path.name} ({bm25_path.stat().st_size/1e6:.1f}MB), {meta.name}", flush=True)

        # 自检：用 dev 的前 5 句查 top3，人工可判断相关性
        dev = DATA / "mtnt.dev.jsonl"
        if dev.exists() and args.pool == "seen":
            with dev.open(encoding="utf-8") as fh:
                probes = [json.loads(l) for l in fh.readlines()[:5]]
            q = encode([p["src"] for p in probes])
            D, I = index.search(q, 3)
            print("\n[自检] MTNT dev 前 5 句的 dense top-3：", flush=True)
            for p, scores, ids in zip(probes, D, I):
                print(f"  Q: {p['src'][:70]}", flush=True)
                for s, i in zip(scores, ids):
                    print(f"     {s:.3f}  {rows[i]['src'][:60]} || {rows[i]['tgt'][:60]}", flush=True)

    main(args)

run_stage('index', 'Cell D2: retrieval index',
          fn=lambda: stage_index(SimpleNamespace(**{'pool': 'seen', 'limit': None})),
          outputs=('data/index/seen.faiss',))

[skip] Cell D2: retrieval index: RUN['index'] is False


False

### Cell D3: terminology alignment

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import math
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

def stage_glossary_align(args):
    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    OUT = DATA

    EN_TOK = re.compile(r"[A-Za-z][A-Za-z\-']*")
    FR_TOK = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿŒœÆæ][A-Za-zÀ-ÖØ-öø-ÿŒœÆæ\-']*")

    STOP = set("""a an the of in on at to for with and or by from as is are was were be been being
    this that these those it its their his her our your my we you they he she i not no than then
    which who whom whose what when where why how all any both each few more most other some such
    only own same so too very can will just should now during after before above below between
    into through against under over out up down off again further once here there also may might
    must shall would could per via e g ie eg vs etc page emea""".split())

    # EMEA 里高频出现但不构成术语的样板词
    BOILER = set("""european medicines agency human use product information annex summary
    characteristics marketing authorisation holder medicinal product refer to section see
    should be not known frequency very common uncommon rare adverse reaction patients treatment""".split())

    # PDF 页眉页脚碎片：出现在术语里一定是抽取噪声
    PDF_NOISE = {"article", "total", "leaflet", "page", "pages", "date", "version",
                 "printed", "contents", "footer", "header", "updated", "last", "renewed",
                 "contains", "each", "chaque", "contient", "objectives",
                 "january", "february", "march", "april", "may", "june", "july", "august",
                 "september", "october", "november", "december"}

    # 说明书指令性套话：整条丢弃
    BOILER_VERB = {"take", "use", "used", "store", "keep", "tell", "ask", "stop", "call", "read",
                   "check", "wait", "try", "give", "put", "mix", "swallow", "repeat", "see",
                   "select", "choose", "remove", "open", "close", "hold", "insert", "attach",
                   "prepare", "administer", "shake", "wash", "wipe", "discard", "allow", "warm",
                   "renew", "renewed", "inject", "apply", "replace", "dispose", "return", "contact"}
    FR_VERB = {"utilisé", "utiliser", "choisissez", "sélectionnez", "conservez", "prenez", "vérifiez",
               "jetez", "retirez", "appuyez", "tournez", "administrez", "préparez", "rincez",
               "nettoyez", "ouvrez", "fermez", "contactez", "renouvelée", "renouvelé"}
    BOILER_PREFIX = ("your ", "the doctor", "do not", "if you", "you should", "you must",
                     "tell your", "ask your", "before you", "after you", "this medicine",
                     "this product", "the medicine", "the product", "the patient", "it is used")

    # 单位与剂量词（两侧都不算内容词）
    UNIT = set("""mg mcg µg ug ng g kg ml cl dl l iu ui miu mmol mol µm um nm cm mm m
    % cfu kcal kj da kda ppm meq eq bq mbq gb kb mb""".split())

    NUM_RE = re.compile(r"^[\d.,/:\-–—+]+$")
    UNIT_RE = re.compile(r"^(" + "|".join(sorted(map(re.escape, UNIT), key=len, reverse=True)) +
                         r")(/(ml|kg|g|l|m2|cm2|dose|day|h|min))?$", re.I)

    FR_STOP = set("""de du des la le les l d un une et ou pour dans sur avec par au aux en qu que qui
    quoi dont est sont être ont cette ce ces son sa ses leur leurs il elle ils elles on nous vous je tu
    plus moins très tout tous toute toutes même aussi comme lors chez sans sous entre selon après avant
    pendant depuis vers contre y ne pas se c n m t j s""".split())

    EDGE = set("""of the and a an to for with in on by from was were is are be been that this
    these those as at or nor but if then than which who whom whose what when where why how""".split())

    # 法语省音字母（被 tokenizer 切开后需要粘回）
    ELISION = re.compile(r"\b(d|l|n|m|t|s|c|j|qu)\s+([a-zà-ÿ])")
    POSSESS = re.compile(r"\b([A-Z][a-z]{2,})\s+s\s+(?=[a-z])")


    def load(name: str) -> list[dict]:
        with (DATA / name).open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def normalize_text(text: str, side: str) -> str:
        """在分词前修正 PDF 抽取造成的撇号断裂。

        法语：`d ' injection` / `d 'élimination` -> `d'injection`
        英语：`Kaposi ' s` / `Kaposi s sarcoma` -> `Kaposi's`
        按语言分别处理：法语 `s'` 合法（s'il），英语 `X s Y` 是领属格。
        """
        s = text.replace("\u2019", "'").replace("\u00a0", " ")
        if side == "fr":
            s = re.sub(r"\b([dlmntcj])\s*'\s*", r"\1'", s)
        else:
            s = re.sub(r"([A-Za-z])\s+'\s*s\b", r"\1's", s)
            s = re.sub(r"\b([A-Z][a-z]{2,})\s+s\s+(?=[a-z])", r"\1's ", s)
        return re.sub(r"[ \t]+", " ", s)


    def repair(surface: str) -> str:
        """展示用：只做空白与首尾清理，不再改撇号（改在 normalize_text 里做）。"""
        return re.sub(r"\s+", " ", surface).strip()


    def clean_surface(surface: str) -> str:
        """展示串里去掉单位与裸数字：'Advagraf mg gélules' -> 'Advagraf gélules'。"""
        toks = [t for t in surface.split()
                if not UNIT_RE.match(t.strip(".,;:")) and not NUM_RE.match(t.strip(".,;:"))]
        return re.sub(r"\s+", " ", " ".join(toks)).strip()


    BRAND_RE = re.compile(r"^[A-Z]{5,}$")


    def looks_brand(en: str) -> bool:
        """商标/公司名：全大写长串，或每个词都首字母大写且没有小写词（Nobilis Influenza H）。"""
        toks = [t for t in re.findall(r"[A-Za-z][A-Za-z\-']*", en) if not UNIT_RE.match(t)]
        if not toks:
            return True
        if any(BRAND_RE.match(t) for t in toks):                 # ADARTREL / HEXAL / SIFROL
            return True
        if len(toks) == 1:
            t = toks[0]
            if t.isupper():                                      # B-CLL / LLC 这类缩写保留
                return False
            return t[:1].isupper() and len(t) > 3
        return all(t[:1].isupper() for t in toks) and not any(t.islower() for t in toks)


    MED_SUFFIX = ("itis", "osis", "aemia", "emia", "pathy", "plasty", "ectomy", "otomy", "ase",
                  "ium", "one", "oma", "ics", "ism", "mab", "nib", "vir", "mycin", "cillin",
                  "pril", "sartan", "statin", "azole", "tidine", "prazole", "olol", "dipine",
                  "ate", "ide", "gen", "toxin", "feron", "kin", "leukin")


    def worth_single_word(word: str) -> bool:
        """单个词是否值得进术语表：缩写、或带医学术语/药类词尾。

        通用名词（weight / powder / ointment / response）不构成术语，会污染命中率。
        """
        w = word.lower()
        if word.isupper() and 2 <= len(word) <= 6:
            return True
        return any(w.endswith(s) and len(w) > len(s) + 2 for s in MED_SUFFIX)


    def best_surface(counter: Counter) -> str:
        """同一 signature 的多个表面串里挑一个展示：优先非全大写、首字母大写的写法。"""
        if not counter:
            return ""
        ranked = sorted(counter.items(), key=lambda kv: (
            kv[1],                                   # 频次优先
            not kv[0][:1].isupper(),                 # 首字母大写更好
            kv[0].isupper(),                         # 全大写最差
        ), reverse=True)
        return ranked[0][0]



    def content_tokens(span: str, side: str) -> list[str]:
        toks = (EN_TOK if side == "en" else FR_TOK).findall(span)
        out = []
        for t in toks:
            tl = t.lower().strip("'-")
            if not tl or NUM_RE.match(tl) or UNIT_RE.match(tl):
                continue
            if tl in STOP or tl in BOILER or (side == "fr" and tl in FR_STOP):
                continue
            out.append(tl)
        return out


    def signature(span: str, side: str) -> tuple[str, ...]:
        ct = content_tokens(span, side)
        return tuple(sorted(set(ct))) if ct else ()


    def good_span(span: str, side: str = "en") -> bool:
        """掐掉被切碎的短语（Nature and / de benzalkonium）与纯单位串。"""
        toks = span.lower().split()
        if not toks:
            return False
        if UNIT_RE.match(toks[0]) or UNIT_RE.match(toks[-1]):
            return False
        edge = EDGE | (FR_STOP if side == "fr" else set())
        return toks[0] not in edge and toks[-1] not in edge


    def ngrams(tokens: list[str], n_max: int) -> list[str]:
        out = []
        for n in range(1, n_max + 1):
            for i in range(len(tokens) - n + 1):
                g = " ".join(tokens[i:i + n])
                if len(g) <= 60:
                    out.append(g)
        return out


    SEG_SPLIT = re.compile(r"[/(),;:\[\]{}|]|\s[-–—]\s")


    def build_index(rows: list[dict], side: str, n_max: int = 3) -> dict:
        """按内容词 signature 聚合：{sig: {"pairs": set, "surfaces": Counter}}。

        n-gram 只在"分隔符切出的词段"内部生成。EMEA 里 `body weight/day`、`mg/ml`、
        `and/or` 这类用斜杠/括号连写的结构，一旦按整句分词就会拼出 `body weight day`
        这种语料里不存在的假短语（2,029 对含斜杠）。
        """
        tok_re = EN_TOK if side == "en" else FR_TOK
        idx: dict[tuple, dict] = defaultdict(lambda: {"pairs": set(), "surfaces": Counter()})
        for i, r in enumerate(rows):
            text = normalize_text(r["src"] if side == "en" else r["tgt"], side)
            for seg in SEG_SPLIT.split(text):
                toks = tok_re.findall(seg)
                for g in ngrams(toks, n_max):
                    if not good_span(g, side):
                        continue
                    sig = signature(g, side)
                    if not sig:
                        continue
                    if not any(len(t) >= 4 for t in sig):
                        continue
                    idx[sig]["pairs"].add(i)
                    idx[sig]["surfaces"][repair(g)] += 1    # 保留原始大小写，展示时再挑
        return idx


    def mode_align(args) -> None:
        rows = load("emea.train.jsonl")
        print(f"[align] EMEA train {len(rows)} 对", flush=True)

        en_idx = build_index(rows, "en")
        fr_idx = build_index(rows, "fr")
        print(f"  en signature {len(en_idx)}，fr signature {len(fr_idx)}（已按内容词合并词序）",
              flush=True)

        en_sel = {s: d for s, d in en_idx.items() if args.min_cooc <= len(d["pairs"]) <= args.max_freq}
        fr_sel = {s: d for s, d in fr_idx.items() if args.min_cooc <= len(d["pairs"]) <= args.max_freq}
        print(f"  进入对齐: en {len(en_sel)}，fr {len(fr_sel)}", flush=True)

        pair_fr: dict[int, list[tuple]] = defaultdict(list)
        for s, d in fr_sel.items():
            for i in d["pairs"]:
                pair_fr[i].append(s)

        raw: list[dict] = []
        for en_sig, d in en_sel.items():
            en_pairs = d["pairs"]
            c = Counter()
            for i in en_pairs:
                c.update(pair_fr.get(i, ()))
            for fr_sig, cooc in c.items():
                # 长度不一致的对齐基本是"截断"（carbonic anhydrase || anhydrase），
                # 允许差 2 个内容词以防漏掉 "international non-proprietary name" 这类，
                # 但排序时按长度一致度打折，同一英文只保留一致度最高的法语。
                diff = abs(len(en_sig) - len(fr_sig))
                if diff > 2:
                    continue
                cont_en = cooc / len(en_pairs)
                if cont_en < args.min_containment:
                    continue
                cont_fr = cooc / len(fr_sel[fr_sig]["pairs"])
                score = min(cont_en, cont_fr)
                if score < args.min_containment:
                    continue
                rank = score * (1.0 - 0.20 * diff) * (1.0 + 0.05 * (len(en_sig) + len(fr_sig)))
                raw.append({
                    "en": clean_surface(best_surface(d["surfaces"])),
                    "fr": clean_surface(best_surface(fr_sel[fr_sig]["surfaces"])),
                    "en_sig": list(en_sig), "fr_sig": list(fr_sig),
                    "en_variants": [s for s, _ in d["surfaces"].most_common(4)],
                    "fr_variants": [s for s, _ in fr_sel[fr_sig]["surfaces"].most_common(4)],
                    "cooc": cooc, "en_freq": len(en_pairs), "fr_freq": len(fr_sel[fr_sig]["pairs"]),
                    "cont_en": round(cont_en, 3), "cont_fr": round(cont_fr, 3),
                    "score": round(score, 3), "rank": round(rank, 3), "len_diff": diff,
                    "pmi": round(math.log(cooc * len(rows) / (len(en_pairs) * len(fr_sel[fr_sig]["pairs"]))), 2),
                })
        print(f"  原始候选 {len(raw)}", flush=True)

        audit = {}

        # 四个谓词都是保留语义：返回 True 才留下
        def distinct_content(d):
            """内容词集合完全相同（词序/大小写不同）不是翻译，丢掉。"""
            return tuple(d["en_sig"]) != tuple(d["fr_sig"])

        def has_sig(d):
            return bool(d["en_sig"]) and bool(d["fr_sig"])

        def translated(d):
            if abs(len(d["en_sig"]) - len(d["fr_sig"])) > 2:
                return False
            from difflib import SequenceMatcher
            return SequenceMatcher(None, d["en"].lower(), d["fr"].lower()).ratio() < 0.82

        def not_boiler(d):
            en = d["en"]
            if en in BOILER_VERB:
                return False
            if any(t in BOILER_VERB for t in en.lower().split()):  # "solution Select" 这类
                return False
            if any(t in PDF_NOISE for t in en.split()):
                return False
            return not any(en.startswith(p) for p in BOILER_PREFIX)

        def clean_entry(d):
            if any(len(t) <= 1 for t in d["en_sig"]):             # 掩码占位 "x film-coated tablets"
                return False
            if len(d["en_sig"]) == 1 and not worth_single_word(d["en"]):   # 通用单词不是术语
                return False
            if looks_brand(d["en"]):                              # 商标/公司名
                return False
            fr_toks = content_tokens(d["fr"], "fr")
            if any(t in BOILER_VERB or t in FR_VERB for t in fr_toks):   # 混排/指令残片
                return False
            if any(BRAND_RE.match(t) for t in d["fr"].split()):          # 右侧带商标（ABILIFY 这类）
                return False
            return True

        for name, fn in (("词序/大小写相同（非翻译）", distinct_content),
                         ("纯单位或数字", has_sig),
                         ("直译雷同", translated),
                         ("说明书套话", not_boiler),
                         ("专名/掩码/混排残片", clean_entry)):
            before = len(raw)
            raw = [d for d in raw if fn(d)]
            audit[name] = before - len(raw)
            print(f"  过滤 {name}: -{audit[name]}", flush=True)

        # 排序：长度一致度打折后的 rank 优先，其次共现，其次总内容词数（信息量更大）
        raw.sort(key=lambda d: (-d["rank"], -d["cooc"],
                                -(len(d["en_sig"]) + len(d["fr_sig"]))))

        # 一侧多译只留最优：en_sig -> 最佳 fr_sig；fr_sig -> 最佳 en_sig
        best_by_en: dict[tuple, dict] = {}
        for d in raw:
            best_by_en.setdefault(tuple(d["en_sig"]), d)
        best_by_fr: dict[tuple, dict] = {}
        for d in sorted(best_by_en.values(),
                        key=lambda x: (-x["rank"], -x["cooc"], -len(x["fr_sig"]))):
            best_by_fr.setdefault(tuple(d["fr_sig"]), d)
        one2one = sorted(best_by_fr.values(),
                         key=lambda d: (-d["rank"], -d["cooc"],
                                        -(len(d["en_sig"]) + len(d["fr_sig"]))))
        audit["同英多译/同法多源去重"] = len(best_by_en) - len(one2one)
        print(f"  一侧多译去重: {len(best_by_en)} -> {len(one2one)}"
              f"（-{audit['同英多译/同法多源去重']}）", flush=True)

        # 5) 子集去重：X 被 X Y 覆盖时丢 X
        kept: list[dict] = []
        for d in one2one:
            covered = any(
                k is not d and
                (set(d["en_sig"]) < set(k["en_sig"]) or set(d["fr_sig"]) < set(k["fr_sig"])) and
                k["cooc"] >= d["cooc"] * 0.5
                for k in kept)
            if not covered:
                kept.append(d)
        audit["被更长词条覆盖的子集"] = len(one2one) - len(kept)
        print(f"  子集去重: {len(one2one)} -> {len(kept)}（-{audit['被更长词条覆盖的子集']}）", flush=True)

        kept = kept[:args.top]

        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
        for d in kept:
            d["variants"] = []
            for variant in (d["fr"], d["fr"][:1].upper() + d["fr"][1:]):
                ids = tok(" " + variant, add_special_tokens=False)["input_ids"]
                d["variants"].append({"text": variant, "ids": ids, "n_tokens": len(ids),
                                      "single_token": len(ids) == 1})

        path = OUT / "glossary.emea.candidates.jsonl"
        with path.open("w", encoding="utf-8", newline="\n") as fh:
            for d in kept:
                fh.write(json.dumps(d, ensure_ascii=False) + "\n")

        single = sum(1 for d in kept if d["variants"][0]["single_token"])
        audit["输出条数"] = len(kept)
        audit["其中法语单 token"] = single
        (OUT / "glossary.emea.audit.json").write_text(
            json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")

        print(f"\n  输出 {len(kept)} 条（法语单 token {single} 条） -> {path}", flush=True)
        print("  过滤明细: " + json.dumps(audit, ensure_ascii=False), flush=True)
        for d in kept[:args.show]:
            print(f"  {d['score']:.2f} cooc={d['cooc']:>4} {d['en'][:40]:40s} || {d['fr'][:36]:36s}"
                  f" tok={d['variants'][0]['n_tokens']}", flush=True)


    def mode_audit(args) -> None:
        """体检现有候选表：单位、重复、词序、长度。"""
        path = OUT / "glossary.emea.candidates.jsonl"
        rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        print(f"[audit] {path.name}：{len(rows)} 条", flush=True)

        def has_unit(d, side):
            toks = (d["en"] if side == "en" else d["fr"]).split()
            return any(UNIT_RE.match(t.strip(".,")) for t in toks)

        print(f"  英文侧含单位/数字: {sum(1 for d in rows if has_unit(d, 'en'))}")
        print(f"  法语侧含单位/数字: {sum(1 for d in rows if has_unit(d, 'fr'))}")
        esig = Counter(tuple(d.get("en_sig") or signature(d["en"], "en")) for d in rows)
        fsig = Counter(tuple(d.get("fr_sig") or signature(d["fr"], "fr")) for d in rows)
        print(f"  英文 signature 重复组: {sum(1 for v in esig.values() if v > 1)}"
              f"（涉及 {sum(v for v in esig.values() if v > 1)} 条）")
        print(f"  法语 signature 重复组: {sum(1 for v in fsig.values() if v > 1)}"
              f"（涉及 {sum(v for v in fsig.values() if v > 1)} 条）")
        multi = [d for d in rows if len(d["fr"].split()) > 4]
        print(f"  法语侧 >4 词的超长条目: {len(multi)}")
        for d in multi[:5]:
            print(f"    {d['en'][:34]:34s} || {d['fr'][:50]}")
        if args.show:
            print("\n  前 20 条：")
            for d in rows[:args.show]:
                print(f"    {d['score']:.2f} {d['en'][:38]:38s} || {d['fr'][:36]}")


    def mode_slang(args) -> None:
        """跨域对比：在 MTNT 里高频、在 EMEA/clean 里几乎不出现的短 token。"""
        mtnt = load("mtnt.train.jsonl")
        emea = load("emea.train.jsonl")
        clean = load("clean.train.jsonl")

        def counts(rows):
            c = Counter()
            for r in rows:
                c.update(t.lower() for t in EN_TOK.findall(r["src"]))
            return c

        cm, ce, cc = counts(mtnt), counts(emea), counts(clean)
        n_m = len(mtnt)

        FRAG = {"ll", "re", "ve", "s", "t", "d", "m", "don", "doesn", "isn", "wasn", "aren",
                "didn", "won", "can", "couldn", "wouldn", "shouldn", "haven", "hasn", "hadn"}
        SEED = {"tbh", "imo", "imho", "idk", "irl", "btw", "afaik", "smh", "ngl", "rn", "af",
                "lol", "lmao", "rofl", "wtf", "omg", "tmi", "fyi", "iirc", "op", "eli5",
                "tl", "dr", "til", "mfw", "tfw", "jk", "np", "ty", "gg", "ez", "asap",
                "dm", "pm", "ama", "nsfw", "ffs", "stfu", "gtfo", "ikr", "tbf", "ngl"}

        def is_slangish(tok_: str) -> bool:
            if tok_ in SEED:
                return True
            if tok_.upper() == tok_ and 2 <= len(tok_) <= 5:
                return True
            return not re.search(r"[aeiou]", tok_)

        cand = []
        for tok_, n in cm.items():
            if n < args.min_cooc or len(tok_) < 2 or tok_ in STOP or tok_ in FRAG:
                continue
            if not re.fullmatch(r"[a-z][a-z'\-]*", tok_):
                continue
            if not is_slangish(tok_):
                continue
            if (ce.get(tok_, 0) + cc.get(tok_, 0)) / n_m > 0.02:
                continue
            cand.append({"token": tok_, "mtnt_freq": n, "emea_freq": ce.get(tok_, 0),
                         "clean_freq": cc.get(tok_, 0),
                         "in_seed": tok_ in SEED})

        cand.sort(key=lambda d: (not d["in_seed"], -d["mtnt_freq"]))
        cand = cand[:args.top]

        path = OUT / "glossary.mtnt.candidates.jsonl"
        with path.open("w", encoding="utf-8", newline="\n") as fh:
            for d in cand:
                fh.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"[slang] MTNT train {n_m} 句 -> 候选 {len(cand)}（种子表内 "
              f"{sum(1 for d in cand if d['in_seed'])} 条）-> {path}", flush=True)
        for d in cand[:args.show]:
            print(f"  {d['token']:14s} mtnt={d['mtnt_freq']:>4} emea={d['emea_freq']:>3} "
                  f"clean={d['clean_freq']:>3} {'[seed]' if d['in_seed'] else ''}", flush=True)


    def mode_llm(args) -> None:
        """LLM 抽取（召回补充）：本地 Qwen 逐对抽术语，与统计结果交叉验证。"""
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        rows = load("emea.train.jsonl")
        rng = __import__("random").Random(42)
        sample = rng.sample(rows, min(args.samples, len(rows)))
        print(f"[llm] 抽 {len(sample)} 对做术语抽取", flush=True)

        tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
        tok.padding_side = "left"
        tok.pad_token_id = tok.pad_token_id or tok.eos_token_id
        model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-1.5B-Instruct", dtype=torch.bfloat16,
            attn_implementation="sdpa").cuda().eval()

        SYS = ("You extract bilingual terminology from a pharmaceutical document. "
               "Return ONLY a JSON array, no prose. Each item: "
               '{"en": "<English term>", "fr": "<French term>"}. '
               "Keep only domain-specific terms: diseases, drug classes, anatomy, "
               "regulatory or pharmaceutical procedures. Exclude brand names, company "
               "names, units and doses (mg, ml, IU), page numbers, and ordinary vocabulary. "
               "At most 4 items.")

        def prompt(r):
            return f"{SYS}\n\nEN: {r['src'][:400]}\nFR: {r['tgt'][:400]}\n\nJSON:"

        counts: Counter = Counter()
        fails = 0

        def parse(text: str) -> list[dict]:
            i, j = text.find("["), text.rfind("]")
            if i >= 0 and j > i:
                try:
                    obj = json.loads(text[i:j + 1])
                    return [d for d in obj if isinstance(d, dict) and d.get("en") and d.get("fr")]
                except Exception:  # noqa: BLE001
                    pass
            return [{"en": m.group(1), "fr": m.group(2)} for m in
                    re.finditer(r'"en"\s*:\s*"([^"]{2,50})"\s*,\s*"fr"\s*:\s*"([^"]{2,60})"', text)]

        for start in range(0, len(sample), 16):
            chunk = sample[start:start + 16]
            enc = tok([prompt(r) for r in chunk], return_tensors="pt", padding=True).to("cuda")
            with torch.inference_mode():
                out = model.generate(**enc, max_new_tokens=200, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for r, t in zip(chunk, texts):
                pairs = parse(t)
                if not pairs:
                    fails += 1
                for d in pairs:
                    counts[(d["en"].strip().lower(), d["fr"].strip().lower())] += 1
            print(f"  {start + len(chunk)}/{len(sample)}", flush=True)

        ranked = [{"en": en, "fr": fr, "votes": v} for (en, fr), v in counts.most_common()
                  if not any(UNIT_RE.match(t) for t in en.split())]
        path = OUT / "glossary.emea.llm.jsonl"
        with path.open("w", encoding="utf-8", newline="\n") as fh:
            for d in ranked:
                fh.write(json.dumps(d, ensure_ascii=False) + "\n")
        print(f"[llm] 抽到 {len(ranked)} 条去重术语，解析失败 {fails}/{len(sample)} -> {path}",
              flush=True)

        stat_path = OUT / "glossary.emea.candidates.jsonl"
        if stat_path.exists():
            stat = [json.loads(l) for l in stat_path.open(encoding="utf-8")]
            stat_en = {tuple(d.get("en_sig") or signature(d["en"], "en")) for d in stat}
            hit = sum(1 for d in ranked if signature(d["en"], "en") in stat_en)
            print(f"[cross] LLM 术语的英文 signature 与统计表重合 {hit}/{len(ranked)}", flush=True)


    def mode_coverage(args) -> None:
        emea_terms = [json.loads(l) for l in
                      (OUT / "glossary.emea.candidates.jsonl").open(encoding="utf-8")][:args.top]
        mtnt_terms = [json.loads(l) for l in
                      (OUT / "glossary.mtnt.candidates.jsonl").open(encoding="utf-8")][:args.top]

        print(f"[coverage] EMEA {len(emea_terms)} 条 / MTNT {len(mtnt_terms)} 条候选", flush=True)
        for split in ("dev", "test"):
            rows = load(f"emea.{split}.jsonl")
            pats = [re.compile(re.escape(t["en"].lower())) for t in emea_terms]
            covered = sum(1 for r in rows if any(p.search(r["src"].lower()) for p in pats))
            print(f"  emea.{split}: {covered}/{len(rows)} 句"
                  f"（{100 * covered / len(rows):.1f}%）", flush=True)
        for split in ("dev", "test"):
            rows = load(f"mtnt.{split}.jsonl")
            pats = [re.compile(r"(^|\W)" + re.escape(t["token"].lower()) + r"(\W|$)")
                    for t in mtnt_terms]
            covered = sum(1 for r in rows if any(p.search(r["src"].lower()) for p in pats))
            print(f"  mtnt.{split}: {covered}/{len(rows)} 句"
                  f"（{100 * covered / len(rows):.1f}%）", flush=True)


    def mode_prune(args) -> None:
        """按"评测可达性"剪枝：dev 与 test 都不出现的条目对约束解码和命中率都无贡献。

        保留全量到 *.full.jsonl 以便需要时回滚。
        """
        path = OUT / "glossary.emea.candidates.jsonl"
        full = OUT / "glossary.emea.candidates.full.jsonl"
        rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        if not full.exists():
            full.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows),
                            encoding="utf-8")

        dev = [r["src"].lower() for r in load("emea.dev.jsonl")]
        test = [r["src"].lower() for r in load("emea.test.jsonl")]

        kept, dropped = [], 0
        for r in rows:
            en = r["en"].lower()
            r["dev_hits"] = sum(1 for s in dev if en in s)
            r["test_hits"] = sum(1 for s in test if en in s)
            if r["dev_hits"] + r["test_hits"] == 0:
                dropped += 1
                continue
            kept.append(r)

        with path.open("w", encoding="utf-8", newline="\n") as fh:
            for r in kept:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[prune] {len(rows)} -> {len(kept)}（剔除 dev/test 都不出现的 {dropped} 条）",
              flush=True)
        print(f"  全量备份: {full.name}", flush=True)
        print(f"  其中 test 有命中 {sum(1 for r in kept if r['test_hits'] > 0)} 条，"
              f"仅 dev 命中 {sum(1 for r in kept if r['test_hits'] == 0)} 条", flush=True)


    def main(args):
        {"align": mode_align, "slang": mode_slang, "llm": mode_llm,
         "coverage": mode_coverage, "audit": mode_audit, "prune": mode_prune}[args.mode](args)

    main(args)

run_stage('glossary', 'Cell D3: terminology alignment',
          fn=lambda: stage_glossary_align(SimpleNamespace(**{'mode': 'align', 'min_cooc': 6, 'max_freq': 300, 'min_containment': 0.5, 'top': 600, 'samples': 120, 'show': 20})),
          outputs=('data/processed/glossary.emea.candidates.jsonl',))

[skip] Cell D3: terminology alignment: RUN['glossary'] is False


False

### Cell D4: finalise the glossary

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import re
import sys
from pathlib import Path

def stage_glossary_final(args):
    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    REPORTS = ROOT / "reports"


    def load_reviewed(path: Path) -> dict:
        return json.loads(path.read_text(encoding="utf-8"))


    EN_WORD = re.compile(r"[A-Za-z][A-Za-z\-']*")
    FR_WORD = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿŒœÆæ][A-Za-zÀ-ÖØ-öø-ÿŒœÆæ\-']*")
    FR_STOP = set("""de du des la le les l d un une et ou pour dans sur avec par au aux en qu que qui
    quoi dont est sont être ont cette ce ces son sa ses leur leurs il elle ils elles on nous vous je tu
    plus moins très tout tous toute toutes même aussi comme lors chez sans sous entre selon après avant
    pendant depuis vers contre y ne pas se c n m t j s""".split())
    EN_STOP = set("""a an the of in on at to for with and or by from as is are was were be been
    this that these those it its their his her our your my we you they he she i not no than then
    which who whom whose what when where why how all any both each few more most other some such""".split())
    UNIT_TOK = set("""mg mcg µg ug ng g kg ml cl dl l iu ui miu mmol mol µm um nm cm mm m % cfu kcal
    kj da kda ppm meq eq bq mbq gb kb mb""".split())


    def sig_of(text: str, side: str = "en") -> list[str]:
        """内容词签名（排序去重）。命中率匹配用它，词序变化不影响判定。"""
        toks = (EN_WORD if side == "en" else FR_WORD).findall(text)
        out = [t.lower().strip("'-") for t in toks]
        out = [t for t in out if t and t not in UNIT_TOK and not t.isdigit()]
        if side == "fr":
            out = [t for t in out if t not in FR_STOP]
        else:
            out = [t for t in out if t not in EN_STOP]
        return sorted(set(out))


    def variants_of(fr: str) -> list[str]:
        """保守的形态变体：单复数与首字母大小写。宁可少造，不要错配。"""
        out = {fr, fr[0].upper() + fr[1:]}
        if fr.endswith("s"):
            out.add(fr[:-1])
        else:
            out.add(fr + "s")
        if fr.endswith("e"):
            out.add(fr + "s")
        return sorted(out)


    def main(args):

        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

        rev = load_reviewed(Path(args.reviewed))
        emea_rows, mtnt_rows = [], []

        for d in rev.get("emea", []):
            if d.get("decision") not in (1, 2):
                continue
            fr = (d.get("fr_final") or d.get("fr_original") or "").strip()
            if not fr:
                continue
            ids = tok(" " + fr, add_special_tokens=False)["input_ids"]
            variants = variants_of(fr)
            accepted = sorted(set(variants) | set(d.get("accepted") or []))
            emea_rows.append({
                "id": f"emea_{len(emea_rows):04d}",
                "domain": "emea",
                "src": d["en"].strip(),
                "fr": fr,
                "fr_ids": ids,
                "fr_first_id": ids[0] if ids else None,
                "src_sig": sig_of(d["en"]),
                "fr_sig": sig_of(fr, "fr"),
                "single_token": len(ids) == 1,
                "accepted": accepted,
                "variants": variants,
                "decision": d.get("decision"),
                "score": d.get("score"),
                "cooc": d.get("cooc"),
                "test_hits": d.get("test_hits", 0),
                "provenance": "align+manual",
            })

        for d in rev.get("mtnt", []):
            if d.get("decision") != 1:
                continue
            tok_ = d["token"].strip()
            mode = d.get("mode") or "translate"
            fr = (d.get("fr") or "").strip()
            if mode == "keep":
                fr = tok_
            accepted = sorted({tok_, tok_.upper(), fr, fr[:1].upper() + fr[1:] if fr else fr})
            mtnt_rows.append({
                "id": f"mtnt_{len(mtnt_rows):04d}",
                "domain": "mtnt",
                "src": tok_,
                "fr": fr,
                "mode": mode,
                "accepted": [a for a in accepted if a],
                "mtnt_freq": d.get("mtnt_freq"),
                "test_hits": d.get("test_hits", 0),
                "provenance": "attested+manual",
            })

        # 同一法语术语被多条英文命中时只留影响最大的（防止人工合并后重复）
        by_fr: dict[tuple, dict] = {}
        for r in sorted(emea_rows, key=lambda x: -x.get("test_hits", 0)):
            by_fr.setdefault(tuple(r["fr_sig"]), r)
        emea_rows = sorted(by_fr.values(), key=lambda x: -x.get("test_hits", 0))
        for i, r in enumerate(emea_rows):
            r["id"] = f"emea_{i:04d}"

        DATA.mkdir(parents=True, exist_ok=True)
        (DATA / "glossary.emea.json").write_text(
            json.dumps(emea_rows, ensure_ascii=False, indent=1), encoding="utf-8")
        (DATA / "glossary.mtnt.json").write_text(
            json.dumps(mtnt_rows, ensure_ascii=False, indent=1), encoding="utf-8")

        # 覆盖率：有多少 test 句能被术语表"碰到"（决定命中率指标的分母）
        summary = {"emea_terms": len(emea_rows), "mtnt_terms": len(mtnt_rows),
                   "emea_single_token": sum(1 for r in emea_rows if r["single_token"]),
                   "mtnt_modes": {m: sum(1 for r in mtnt_rows if r["mode"] == m)
                                  for m in ("keep", "translate", "omit")}}
        for domain, terms in (("emea", emea_rows), ("mtnt", mtnt_rows)):
            path = DATA / f"{domain}.test.jsonl"
            if not path.exists() or not terms:
                continue
            rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
            if domain == "emea":
                pats = [(t["id"], re.compile(re.escape(t["src"].lower()))) for t in terms]
            else:
                pats = [(t["id"], re.compile(r"(^|\W)" + re.escape(t["src"].lower()) + r"(\W|$)"))
                        for t in terms]
            covered, total_hits = 0, 0
            per_term = {tid: 0 for tid, _ in pats}
            for r in rows:
                src = r["src"].lower()
                hit = False
                for tid, p in pats:
                    if p.search(src):
                        per_term[tid] += 1
                        hit = True
                covered += int(hit)
            total_hits = sum(per_term.values())
            summary[f"{domain}_test"] = {"rows": len(rows), "covered": covered,
                                         "coverage_pct": round(100 * covered / max(1, len(rows)), 1),
                                         "term_hits": total_hits}
            # 把 test 命中数回写到术语表，便于报告里按影响排序
            if domain == "emea":
                for t in emea_rows:
                    t["test_hits"] = per_term.get(t["id"], 0)
            else:
                for t in mtnt_rows:
                    t["test_hits"] = per_term.get(t["id"], 0)

        (DATA / "glossary.emea.json").write_text(
            json.dumps(emea_rows, ensure_ascii=False, indent=1), encoding="utf-8")
        (DATA / "glossary.mtnt.json").write_text(
            json.dumps(mtnt_rows, ensure_ascii=False, indent=1), encoding="utf-8")
        (REPORTS / "glossary_summary.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print(f"\n-> {DATA/'glossary.emea.json'}")
        print(f"-> {DATA/'glossary.mtnt.json'}")
        print(f"-> {REPORTS/'glossary_summary.json'}")

    main(args)

run_stage('glossary', 'Cell D4: finalise the glossary',
          fn=lambda: stage_glossary_final(SimpleNamespace(**{'reviewed': None, 'min_decision': 1})),
          outputs=('data/processed/glossary.emea.json',))

[skip] Cell D4: finalise the glossary: RUN['glossary'] is False


False

### Cell D5: augmentation arm

Forward translation, back-translation and the WMT24 chat pairs are mixed into
`data/processed/sft.train.augmented.jsonl`; Cell T5 then trains on it. Only needed for
the +aug row of Table 1.

In [ ]:
# 增强语料：chat（WMT24 人写对话）-> back（Opus-MT 回译）-> forward（SFT 自译）->
# mix（三路合成 + 真实噪声，写 data/processed/sft.train.augmented.jsonl）。
# 之后用 Cell T5 在增强语料上重训。
if RUN["augment"]:
    for _stage in ("chat", "back", "forward", "mix"):
        run_stage("augment", f"Cell D5: augmentation · {_stage}",
                  fn=lambda s=_stage: _stream([sys.executable, "-u",
                                               "src/data/augment_mtnt.py", "--stage", s]))
else:
    print("[skip] Cell D5: RUN['augment'] is False", flush=True)

[skip] Cell D5: RUN['augment'] is False


Corpora after cleaning. The clean set is a sanity check only: 2,490 of its 2,500 original test sentences also appear in training.

| Corpus | Train | Dev | Test | Register |
| --- | --- | --- | --- | --- |
| clean (synthetic phrases) | 1,800 | 200 | 251 | short, templated |
| MTNT en-fr | 32,593 | 852 | 1,020 | Reddit, noisy |
| EMEA en-fr | 27,000 | 1,000 | 2,000 | regulatory, term-dense |
| WMT24 Chat (added later) | 5,252 | 1,209 | - | customer support |

## 3. Systems

Table 2 summarises the configurations.

The five groups as implemented. G5* is the variant that survived the ablations: the four-module stack is reported as a negative result.

| Group | Configuration | Train. | Note |
| --- | --- | --- | --- |
| G1 | Base LLM, zero-shot | 0 | lower bound |
| G2 | G1 plus dense/BM25 retrieval, 3-shot | 0 | explicit memory |
| G3 | LoRA SFT on 61.4k pairs | 1.18% | main gain |
| G4 | G3 plus retrieval, same prompt as G2 | 1.18% | format control |
| G5 | DPO, RAG, glossary, critic, router | 1.18% | reported as ablation |
| G5* | SFT plus DPO, no context | 1.18% | best measured |

**Glossary.** A 220-entry EMEA term table and a 175-entry MTNT slang table were built by
statistical alignment on the training split, then adjudicated by an external model
(GPT-6 Luna) against the corpus's own sentence pairs, with a second model (Claude Haiku 5.5)
re-checking a stratified sample of 30 items: agreement 90%, two accepted corrections, one
false alarm. Only one EMEA term has a single-token French form, so decoder-level forcing is
impractical. The glossary is used as a prompt hint and as the denominator of the hit-rate
metric.

**Routing.** A two-pass design separates generation from scoring: pass 1 emits both the light
draft and the heavy-path output, pass 2 scores each with CometKiwi, pass 3 applies the threshold
`tau` offline as a filter. This makes a threshold sweep free, and it keeps the quality-estimation
model out of the generation process.

## 4. Evaluation protocol

**Metrics.** COMET-DA (`wmt22-comet-da`) is the primary metric, reported per domain with the
two-domain macro average as the headline. BLEU and chrF++ are corpus-level. A terminology hit
rate is computed on sentences whose source contains a glossary entry, using an accepted-variant
set per entry so that inflection and word order do not count as misses. Latency is batch-1 p50
in milliseconds; throughput is batch-32 sentences per second.

**Statistics.** Per-sentence scores are stored, and comparisons use a paired bootstrap with
10,000 resamples. For MTNT, resampling is clustered by Reddit comment id because sentences from
one comment are not independent. p-values are Holm-corrected within each family. The minimum
detectable effect at n = 1,020 is 0.012 COMET-DA; differences below that are reported as
"below resolution" rather than as improvements.

**Pre-registered rules.** Retrieval depth was chosen on dev by two-domain macro average
(k=3, tied with k=5 within 0.0004, chosen for cost). The router threshold is chosen on dev by
"max macro subject to latency <= 1.15x that of G4". Test data was used once.

### Cell T1: LoRA SFT

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import sys
import time
from pathlib import Path

def stage_sft(args):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    import torch  # noqa: E402

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    RUNS = ROOT / "runs"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    SYSTEM = ("You are a professional English-to-French translator. Translate the user's sentence "
              "into natural, accurate French. Output only the translation.")
    GB = 1e9


    def load(name: str) -> list[dict]:
        with (DATA / name).open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def to_messages(rows: list[dict]) -> list[dict]:
        return [{"messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": r["src"]},
            {"role": "assistant", "content": r["tgt"]},
        ]} for r in rows]


    def main(args):

        from datasets import Dataset
        from peft import LoraConfig
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from trl import SFTConfig, SFTTrainer

        torch.cuda.set_per_process_memory_fraction(0.95)

        if args.train_file:
            path = Path(args.train_file)
            if not path.is_absolute():
                path = DATA / args.train_file
            with path.open(encoding="utf-8") as fh:
                train_rows = [json.loads(l) for l in fh if l.strip()]
            print(f"[data] 使用 {path.name}：{len(train_rows)} 条", flush=True)
        else:
            train_rows = (load("mtnt.train.jsonl") + load("emea.train.jsonl")
                          + load("clean.train.jsonl"))
        if args.limit:
            train_rows = (train_rows[: args.limit // 3] +
                          load("emea.train.jsonl")[: args.limit // 3] +
                          load("clean.train.jsonl")[: args.limit // 3])
        eval_rows = load("mtnt.dev.jsonl")[: args.eval_limit // 2] + \
                    load("emea.dev.jsonl")[: args.eval_limit // 2]
        print(f"[data] train {len(train_rows)}，eval {len(eval_rows)}", flush=True)

        out_dir = RUNS / args.run_id
        out_dir.mkdir(parents=True, exist_ok=True)

        tok = AutoTokenizer.from_pretrained(MODEL)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, dtype=torch.bfloat16, attn_implementation="sdpa")

        lora = LoraConfig(
            r=args.lora_r, lora_alpha=2 * args.lora_r, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            task_type="CAUSAL_LM",
        )

        steps_per_epoch = max(1, len(train_rows) // (args.batch * args.accum))
        cfg = SFTConfig(
            output_dir=str(out_dir),
            max_length=args.max_length,
            per_device_train_batch_size=args.batch,
            gradient_accumulation_steps=args.accum,
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            lr_scheduler_type="cosine",
            warmup_steps=max(10, int(0.03 * steps_per_epoch * args.epochs)),
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            packing=False,
            assistant_only_loss=True,
            optim="adamw_torch",
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            torch_empty_cache_steps=100,
            logging_steps=20,
            eval_strategy="steps",
            eval_steps=400,
            per_device_eval_batch_size=args.batch,
            save_strategy="steps",
            save_steps=400,
            save_total_limit=2,
            report_to=[],
            run_name=args.run_id,
            seed=args.seed,
            use_liger_kernel=False,
            torch_compile=False,
        )

        trainer = SFTTrainer(
            model=model, args=cfg,
            train_dataset=Dataset.from_list(to_messages(train_rows)),
            eval_dataset=Dataset.from_list(to_messages(eval_rows)),
            peft_config=lora, processing_class=tok,
        )
        trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in trainer.model.parameters())
        print(f"[lora] trainable {trainable/1e6:.2f}M / {total/1e6:.2f}M = "
              f"{100*trainable/total:.3f}%", flush=True)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        trainer.train()
        wall = (time.perf_counter() - t0) / 3600

        adapter_dir = out_dir / "adapter"
        trainer.save_model(str(adapter_dir))
        tok.save_pretrained(str(adapter_dir))

        meta = {"run_id": args.run_id, "train_rows": len(train_rows), "eval_rows": len(eval_rows),
                "epochs": args.epochs, "batch": args.batch, "accum": args.accum,
                "max_length": args.max_length, "lr": args.lr, "lora_r": args.lora_r,
                "trainable_M": round(trainable / 1e6, 2),
                "trainable_pct": round(100 * trainable / total, 3),
                "hours": round(wall, 2),
                "peak_vram_gb": round(torch.cuda.max_memory_allocated() / GB, 3),
                "final_train_loss": trainer.state.log_history[-1].get("loss"),
                "steps_per_epoch": steps_per_epoch,
                "git_commit": os.popen("git rev-parse --short HEAD").read().strip() or "n/a"}
        (out_dir / "train_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2),
                                                 encoding="utf-8")
        print("[meta] " + json.dumps(meta, ensure_ascii=False), flush=True)
        print(f"[out] {adapter_dir}", flush=True)

    main(args)

run_stage('sft', 'Cell T1: LoRA SFT',
          fn=lambda: stage_sft(SimpleNamespace(**{'run_id': 'g3_sft_r16_s42', 'epochs': 2, 'batch': 16, 'accum': 1, 'lr': 0.0002, 'max_length': 384, 'lora_r': 16, 'limit': None, 'train_file': None, 'eval_limit': 600, 'seed': 42})),
          outputs=('runs/g3_sft_r16_s42/adapter/adapter_config.json',))

[skip] Cell T1: LoRA SFT: RUN['sft'] is False


False

### Cell T2: mine preference pairs

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import random
import sys
import time
from collections import Counter
from pathlib import Path

def stage_mine(args):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")

    import torch  # noqa: E402

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    INTERIM = ROOT / "data" / "interim"
    RUNS = ROOT / "runs"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    SYSTEM = ("You are a professional English-to-French translator. Translate the user's sentence "
              "into natural, accurate French. Output only the translation.")
    GB = 1e9


    def load(name: str) -> list[dict]:
        with (DATA / name).open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def stage_generate(args) -> None:
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer

        torch.cuda.set_per_process_memory_fraction(0.95)
        rng = random.Random(args.seed)

        mtnt = load("mtnt.train.jsonl")
        emea = load("emea.train.jsonl")
        rng.shuffle(mtnt)
        rng.shuffle(emea)
        prompts = ([{**r, "domain": "mtnt"} for r in mtnt[: args.n_mtnt]] +
                   [{**r, "domain": "emea"} for r in emea[: args.n_emea]])
        rng.shuffle(prompts)
        print(f"[generate] {len(prompts)} 个源句 × K={args.k} 候选", flush=True)

        tok = AutoTokenizer.from_pretrained(MODEL)
        tok.padding_side = "left"
        tok.pad_token_id = tok.pad_token_id or tok.eos_token_id
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, dtype=torch.bfloat16, attn_implementation="sdpa")
        model = PeftModel.from_pretrained(model, args.adapter).merge_and_unload().cuda().eval()

        INTERIM.mkdir(parents=True, exist_ok=True)
        out_path = INTERIM / "candidates.jsonl"
        t0 = time.perf_counter()
        with out_path.open("w", encoding="utf-8", newline="\n") as fh:
            for start in range(0, len(prompts), args.batch):
                chunk = prompts[start:start + args.batch]
                texts = [tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": r["src"]}],
                    tokenize=False, add_generation_prompt=True) for r in chunk]
                enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
                with torch.inference_mode():
                    out = model.generate(**enc, do_sample=True, temperature=0.9, top_p=0.95,
                                         num_return_sequences=args.k, max_new_tokens=args.max_new_tokens,
                                         pad_token_id=tok.pad_token_id)
                gen = out[:, enc["input_ids"].shape[1]:]
                preds = tok.batch_decode(gen, skip_special_tokens=True)
                for i, r in enumerate(chunk):
                    cands = [p.strip() for p in preds[i * args.k:(i + 1) * args.k]]
                    fh.write(json.dumps({"id": r["id"], "domain": r["domain"], "src": r["src"],
                                         "tgt": r["tgt"], "cands": cands},
                                        ensure_ascii=False) + "\n")
                if start % (args.batch * 20) == 0:
                    done = start + len(chunk)
                    rate = done / (time.perf_counter() - t0)
                    print(f"  {done}/{len(prompts)}  {rate:.1f} 句/s  "
                          f"峰值 {torch.cuda.max_memory_allocated()/GB:.2f}GB", flush=True)
        print(f"[generate] -> {out_path} 用时 {(time.perf_counter()-t0)/60:.1f} 分钟", flush=True)


    def stage_pair(args) -> None:
        scored = [json.loads(l) for l in
                  (INTERIM / "candidates.scored.jsonl").open(encoding="utf-8") if l.strip()]
        print(f"[pair] 读入 {len(scored)} 条已打分样本", flush=True)

        pairs, stats = [], Counter()
        for r in scored:
            cands = r["cands"]
            if len(cands) < 2:
                stats["候选不足"] += 1
                continue
            ranked = sorted(cands, key=lambda c: -c["da"])
            best, worst = ranked[0], ranked[-1]
            if best["text"].strip().lower() == worst["text"].strip().lower():
                stats["首尾相同"] += 1
                continue
            if best["da"] - worst["da"] < args.min_delta:
                stats["分差过小"] += 1
                continue
            lb, lw = len(best["text"].split()), len(worst["text"].split())
            if lw == 0 or not (args.ratio_lo <= lb / lw <= args.ratio_hi):
                stats["长度比越界"] += 1
                continue
            pairs.append({"prompt": r["src"], "chosen": best["text"], "rejected": worst["text"],
                          "domain": r["domain"], "delta_da": round(best["da"] - worst["da"], 4),
                          "chosen_da": round(best["da"], 4), "rejected_da": round(worst["da"], 4)})
            stats["保留"] += 1

        rng = random.Random(args.seed)
        rng.shuffle(pairs)
        n_dev = min(2000, len(pairs) // 10)
        dev, train = pairs[:n_dev], pairs[n_dev:]
        DATA.mkdir(parents=True, exist_ok=True)
        for name, rows in (("train", train), ("dev", dev)):
            with (DATA / f"dpo.pairs.{name}.jsonl").open("w", encoding="utf-8", newline="\n") as fh:
                for r in rows:
                    fh.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[pair] 保留 {len(pairs)}（train {len(train)} / dev {len(dev)}）", flush=True)
        print("[pair] 丢弃原因: " + json.dumps(dict(stats), ensure_ascii=False), flush=True)
        deltas = sorted(p["delta_da"] for p in pairs)
        if deltas:
            q = lambda v, p: v[min(len(v) - 1, int(len(v) * p / 100))]
            print(f"[pair] Δ_DA 分位: p10={q(deltas,10):.3f} p50={q(deltas,50):.3f} "
                  f"p90={q(deltas,90):.3f}", flush=True)


    def main(args):
        {"generate": stage_generate, "pair": stage_pair}[args.stage](args)

    main(args)

run_stage('dpo', 'Cell T2: mine preference pairs',
          fn=lambda: stage_mine(SimpleNamespace(**{'stage': 'generate', 'adapter': None, 'n_mtnt': 6000, 'n_emea': 12000, 'k': 4, 'batch': 8, 'max_new_tokens': 160, 'min_delta': 0.1, 'ratio_lo': 0.6, 'ratio_hi': 1.6, 'seed': 42})),
          outputs=('data/interim/candidates.jsonl',))

[skip] Cell T2: mine preference pairs: RUN['dpo'] is False


False

### Cell T3: score the candidates

In [ ]:
from __future__ import annotations

import io
import json
import os
import sys
import time
from pathlib import Path

def stage_score_cand(args):
    ROOT = PROJ
    INTERIM = ROOT / "data" / "interim"
    QE_LOCAL = str(ROOT / "runs" / "qe_local" / "wmt22-cometkiwi-da-mirror" / "checkpoints" / "model.ckpt")


    def load_model(repo: str):
        from comet import download_model, load_from_checkpoint
        path = repo if Path(repo).exists() else download_model(repo)
        return load_from_checkpoint(path)


    def predict(model, data: list[dict], batch_size: int):
        try:
            out = model.predict(data, batch_size=batch_size, gpus=1, progress_bar=False)
        except TypeError:
            out = model.predict(data, batch_size=batch_size, accelerator="gpu", devices=1,
                                progress_bar=False)
        if isinstance(out, tuple):
            return list(out[0])
        return list(getattr(out, "scores", None) or out["scores"])


    def main(args):
        src = INTERIM / "candidates.jsonl"
        dst = INTERIM / "candidates.scored.jsonl"
        rows = [json.loads(l) for l in src.open(encoding="utf-8") if l.strip()]
        n_cand = sum(len(r["cands"]) for r in rows)
        print(f"[score] {len(rows)} 个源句，{n_cand} 个候选", flush=True)

        da = load_model("Unbabel/wmt22-comet-da")
        qe = load_model(QE_LOCAL)
        print("[score] 两个打分模型已加载", flush=True)

        flat = [(i, j, c) for i, r in enumerate(rows) for j, c in enumerate(r["cands"]) if c.strip()]
        print(f"[score] 有效候选 {len(flat)}", flush=True)

        t0 = time.perf_counter()
        da_scores = predict(da, [{"src": rows[i]["src"], "mt": c, "ref": rows[i]["tgt"]}
                                 for i, _, c in flat], batch_size=64)
        print(f"[score] DA 完成 {(time.perf_counter()-t0)/60:.1f} 分钟", flush=True)
        t0 = time.perf_counter()
        qe_scores = predict(qe, [{"src": rows[i]["src"], "mt": c} for i, _, c in flat],
                            batch_size=64)
        print(f"[score] Kiwi 完成 {(time.perf_counter()-t0)/60:.1f} 分钟", flush=True)

        packed: dict[int, list[dict]] = {}
        for (i, _, c), d, q in zip(flat, da_scores, qe_scores):
            packed.setdefault(i, []).append({"text": c, "da": float(d), "kiwi": float(q)})

        with dst.open("w", encoding="utf-8", newline="\n") as fh:
            for i, r in enumerate(rows):
                cands = packed.get(i, [])
                if not cands:
                    continue
                fh.write(json.dumps({**{k: r[k] for k in ("id", "domain", "src", "tgt")},
                                     "cands": cands}, ensure_ascii=False) + "\n")

        # 顺带算候选之间的 DA 离散度与 Kiwi/DA 秩相关（决定偏好对是否够"硬"）
        import statistics
        spreads = [max(c["da"] for c in v) - min(c["da"] for c in v) for v in packed.values()]
        print(f"[score] 组内 DA 极差: 中位 {statistics.median(spreads):.3f}，"
              f">0.10 的组 {sum(1 for s in spreads if s > 0.10)}/{len(spreads)}", flush=True)
        print(f"[score] -> {dst}", flush=True)

    main(args)

run_stage('dpo', 'Cell T3: score the candidates',
          fn=lambda: stage_score_cand(SimpleNamespace(**{})),
          outputs=('data/interim/candidates.scored.jsonl',),
          comet=('src/train/score_candidates.py', {}))

[skip] Cell T3: score the candidates: RUN['dpo'] is False


False

### Cell T4: DPO

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import random
import sys
import time
from pathlib import Path

def stage_dpo(args):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")

    import torch  # noqa: E402

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    RUNS = ROOT / "runs"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    GB = 1e9


    def load(name: str) -> list[dict]:
        with (DATA / name).open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def main(args):

        from datasets import Dataset
        from peft import LoraConfig
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from trl import DPOConfig, DPOTrainer

        torch.cuda.set_per_process_memory_fraction(0.95)

        train = load("dpo.pairs.train.jsonl")
        dev = load("dpo.pairs.dev.jsonl")
        if args.stage_random:
            rng = random.Random(args.seed)
            for r in train:
                if rng.random() < 0.5:
                    r["chosen"], r["rejected"] = r["rejected"], r["chosen"]
            print(f"[random] 已随机交换一半样本的 chosen/rejected（{len(train)} 对）", flush=True)

        print(f"[data] train {len(train)} 对，dev {len(dev)} 对", flush=True)

        out_dir = RUNS / args.run_id
        out_dir.mkdir(parents=True, exist_ok=True)

        tok = AutoTokenizer.from_pretrained(MODEL)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, dtype=torch.bfloat16, attn_implementation="sdpa")
        if args.init_adapter and Path(args.init_adapter).exists():
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, args.init_adapter).merge_and_unload()
            print(f"[init] 已合并 SFT adapter: {args.init_adapter}", flush=True)

        lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                          "gate_proj", "up_proj", "down_proj"])

        steps = max(1, len(train) // (args.batch * args.accum))
        cfg = DPOConfig(
            output_dir=str(out_dir),
            beta=args.beta,
            loss_type="sigmoid",
            label_smoothing=0.0,
            max_length=args.max_length,
            per_device_train_batch_size=args.batch,
            gradient_accumulation_steps=args.accum,
            num_train_epochs=args.epochs,
            learning_rate=args.lr,
            lr_scheduler_type="cosine",
            warmup_steps=max(10, int(0.1 * steps)),
            bf16=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            precompute_ref_log_probs=True,
            precompute_ref_batch_size=8,
            optim="adamw_torch",
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            torch_empty_cache_steps=100,
            logging_steps=20,
            eval_strategy="steps",
            eval_steps=200,
            save_strategy="steps",
            save_steps=200,
            save_total_limit=2,
            report_to=[],
            run_name=args.run_id,
            seed=args.seed,
        )

        trainer = DPOTrainer(
            model=model, ref_model=None, args=cfg,
            train_dataset=Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"],
                                              "rejected": r["rejected"]} for r in train]),
            eval_dataset=Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"],
                                             "rejected": r["rejected"]} for r in dev]),
            peft_config=lora, processing_class=tok,
        )

        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        trainer.train()
        wall = (time.perf_counter() - t0) / 3600

        adapter_dir = out_dir / "adapter"
        trainer.save_model(str(adapter_dir))
        tok.save_pretrained(str(adapter_dir))

        hist = [h for h in trainer.state.log_history if "rewards/margins" in h]
        meta = {"run_id": args.run_id, "pairs": len(train), "beta": args.beta, "lr": args.lr,
                "hours": round(wall, 2),
                "peak_vram_gb": round(torch.cuda.max_memory_allocated() / GB, 3),
                "final_margin": hist[-1]["rewards/margins"] if hist else None,
                "final_accuracy": hist[-1]["rewards/accuracies"] if hist else None,
                "stage_random": args.stage_random,
                "git_commit": os.popen("git rev-parse --short HEAD").read().strip() or "n/a"}
        (out_dir / "train_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2),
                                                 encoding="utf-8")
        print("[meta] " + json.dumps(meta, ensure_ascii=False), flush=True)

    main(args)

run_stage('dpo', 'Cell T4: DPO',
          fn=lambda: stage_dpo(SimpleNamespace(**{'run_id': 'g5_dpo_r16_s42', 'init_adapter': None, 'beta': 0.1, 'lr': 5e-06, 'epochs': 2, 'batch': 8, 'accum': 2, 'max_length': 384, 'stage_random': None, 'seed': 42})),
          outputs=('runs/g5_dpo_r16_s42/adapter/adapter_config.json',))

[skip] Cell T4: DPO: RUN['dpo'] is False


False

### Cell T5: SFT on the augmented corpus

The same SFT code as Cell T1, pointed at the augmented file. Runs after Cell D5.

In [ ]:
# +aug 那一行：同一份 SFT 代码，换训练文件与 run_id。
if RUN["augment"]:
    run_stage("augment", "Cell T5: SFT on the augmented corpus",
              fn=lambda: stage_sft(SimpleNamespace(
                  run_id="g3_sft_aug", epochs=2.0, batch=16, accum=1, lr=0.0002,
                  max_length=384, lora_r=16, limit=None,
                  train_file="data/processed/sft.train.augmented.jsonl",
                  eval_limit=600, seed=42)),
              outputs=("runs/g3_sft_aug/adapter/adapter_config.json",))
else:
    print("[skip] Cell T5: RUN['augment'] is False", flush=True)

[skip] Cell T5: RUN['augment'] is False


## 5. Results

### Cell R1: reproduce every run in the matrix

The narrative path is Cells I1–E2. This driver reproduces the remaining runs (the G4
inline control, the G5 dev pass, the five leave-one-out arms) and then scores and routes
them. It reuses `run_matrix.py`, skips runs whose outputs exist, and streams the child
processes' output as it arrives.

In [ ]:
# 全部 run：g4-inline、g5 dev、五个留一，加打分与路由。
# run_matrix.py 按产物跳过，跑过的组只打印 [skip]，可反复执行。
if RUN["matrix"]:
    run_stage("matrix", "Cell R1: full run matrix (g4 / g5 / ablations)",
              fn=lambda: _stream([sys.executable, "-u", "src/infer/run_matrix.py",
                                  "--group", "all"]))
else:
    print("[skip] Cell R1: RUN['matrix'] is False", flush=True)

[skip] Cell R1: RUN['matrix'] is False


### Cell I1: batch inference

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import pickle
import subprocess
import sys
import time
from pathlib import Path

def stage_infer(args):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    import torch  # noqa: E402

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    INDEX = ROOT / "data" / "index"
    RUNS = ROOT / "runs"

    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    SYSTEM = ("You are a professional English-to-French translator. Translate the user's sentence "
              "into natural, accurate French. Output only the translation.")
    SET_FILES = {
        "test": {"mtnt": "mtnt.test.jsonl", "emea": "emea.test.jsonl", "clean": "clean.test.jsonl"},
        "dev": {"mtnt": "mtnt.dev.jsonl", "emea": "emea.dev.jsonl", "clean": "clean.dev.jsonl"},
    }
    GB = 1e9


    def git_commit() -> str:
        try:
            return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                                  capture_output=True, text=True, timeout=15).stdout.strip() or "n/a"
        except Exception:  # noqa: BLE001
            return "n/a"


    def load_rows(names: list[str], limit: int | None) -> list[dict]:
        rows = []
        for n in names:
            with (DATA / n).open(encoding="utf-8") as fh:
                rows += [json.loads(l) for l in fh if l.strip()]
        return rows[:limit] if limit else rows


    class Retriever:
        """Dense(bge-m3 + FAISS) + BM25，RRF 融合。pool=seen|unseen。"""

        def __init__(self, pool: str = "seen", rrf_k: int = 60):
            import faiss
            from sentence_transformers import SentenceTransformer

            self.rrf_k = rrf_k
            self.index = faiss.read_index(str(INDEX / f"{pool}.faiss"))
            with (INDEX / f"{pool}.bm25.pkl").open("rb") as fh:
                self.bm25 = pickle.load(fh)
            self.meta = [json.loads(l) for l in (INDEX / f"{pool}.meta.jsonl").open(encoding="utf-8")]
            self.enc = SentenceTransformer("BAAI/bge-m3", device="cuda",
                                           model_kwargs={"torch_dtype": torch.float16})
            self.pool = pool

        def search(self, queries: list[str], k: int = 3) -> list[list[dict]]:
            # 过取 4k 再按源句去重：EMEA/MTNT 里同一源句常有多条参考译文，
            # 同一源句常有多条参考译文，不去重会让 3-shot 出现重复示例。
            fetch = max(k * 4, 12)
            q = self.enc.encode(queries, batch_size=64, normalize_embeddings=True,
                                convert_to_numpy=True).astype("float32")
            _, dense_ids = self.index.search(q, fetch)
            out = []
            for qi, query in enumerate(queries):
                scores = self.bm25.get_scores(query.lower().split())
                bm25_ids = scores.argsort()[::-1][:fetch]
                rrf: dict[int, float] = {}
                for rank, i in enumerate(dense_ids[qi]):
                    rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (self.rrf_k + rank + 1)
                for rank, i in enumerate(bm25_ids):
                    rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (self.rrf_k + rank + 1)
                picked, seen_src = [], set()
                for i, _ in sorted(rrf.items(), key=lambda x: -x[1]):
                    key = self.meta[i]["src"].strip().lower()
                    if key in seen_src or key == query.strip().lower():
                        continue
                    seen_src.add(key)
                    picked.append(self.meta[i])
                    if len(picked) == k:
                        break
                out.append(picked)
            return out


    def load_examples(rows: list[dict], pool: str, k: int, same_domain: bool = False) -> list[list[dict]]:
        """带磁盘缓存的检索：同一 (pool, id) 只算一次，所有系统与所有 k 共用。

        缓存固定存 top-12，取用时再切前 k 条——这样 k=1/3/5 的扫描只付一次 BM25 成本
        （61,393 文档上是纯 Python 打分，实测 ~0.21s/句）。
        """
        cache_dir = INDEX / "cache"
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_k = max(k, 12)
        cache_path = cache_dir / f"{pool}.top{cache_k}.jsonl"

        cached: dict[str, list[dict]] = {}
        if cache_path.exists():
            with cache_path.open(encoding="utf-8") as fh:
                for line in fh:
                    if line.strip():
                        d = json.loads(line)
                        cached[d["id"]] = d["examples"]

        missing = [r for r in rows if r["id"] not in cached]
        if missing:
            print(f"[retrieval] 缓存命中 {len(rows) - len(missing)}，需计算 {len(missing)}", flush=True)
            retr = Retriever(pool)
            t0 = time.perf_counter()
            with cache_path.open("a", encoding="utf-8", newline="\n") as fh:
                for start in range(0, len(missing), 64):
                    chunk = missing[start:start + 64]
                    exs = retr.search([r["src"] for r in chunk], k=cache_k)
                    for r, ex in zip(chunk, exs):
                        cached[r["id"]] = ex
                        fh.write(json.dumps({"id": r["id"], "examples": ex}, ensure_ascii=False) + "\n")
            print(f"[retrieval] 完成 {len(missing)} 句，用时 {time.perf_counter() - t0:.1f}s", flush=True)
            del retr
            torch.cuda.empty_cache()
        else:
            print(f"[retrieval] 全部命中缓存（{len(rows)} 句）", flush=True)
        if same_domain:
            # 域门控：MTNT 查询约 12% 会检索到 EMEA 样板句。
            # 缓存里存的是 top-12，足以在同域内重排出前 k 条。
            out = []
            for r in rows:
                ex = cached[r["id"]]
                same = [e for e in ex if e.get("domain") == r["domain"]]
                other = [e for e in ex if e.get("domain") != r["domain"]]
                out.append((same + other)[:k])
            return out
        return [cached[r["id"]][:k] for r in rows]


    def build_messages(src: str, examples: list[dict] | None, fmt: str = "turns") -> list[dict]:
        """两种注入格式。

        turns  : 每个示例占一对 user/assistant 轮次（G2 用的原格式）
        inline : 把示例塞进同一个 user 消息，消息结构仍是 system + 单条 user，
                 与 SFT 训练时完全一致（SFT 只见过单轮，多轮示例会造成训练/推理格式错配）
        """
        if fmt == "inline" and examples:
            lines = ["Reference translations of similar sentences:"]
            for ex in examples:
                lines.append(f"EN: {ex['src']}\nFR: {ex['tgt']}")
            lines.append(f"Translate this sentence:\nEN: {src}\nFR:")
            return [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": "\n\n".join(lines)}]

        msgs = [{"role": "system", "content": SYSTEM}]
        for ex in examples or []:
            msgs.append({"role": "user", "content": ex["src"]})
            msgs.append({"role": "assistant", "content": ex["tgt"]})
        msgs.append({"role": "user", "content": src})
        return msgs


    def main(args):

        from transformers import AutoModelForCausalLM, AutoTokenizer

        torch.cuda.set_per_process_memory_fraction(0.95)
        out_dir = RUNS / args.run_id
        out_dir.mkdir(parents=True, exist_ok=True)

        files = [SET_FILES[args.split][d] for d in args.domains.split(",") if d in SET_FILES[args.split]]
        rows = load_rows(files, args.limit)
        print(f"[data] {args.split} {len(rows)} 句 from {files}", flush=True)

        # 先检索再加载 LLM：bge-m3 用完即释放，避免与 Qwen 叠加占显存
        examples_all = (load_examples(rows, args.pool, args.k, args.domain_gated)
                        if args.mode in ("rag", "sft_rag") else None)
        retrieved_log = ([{"id": r["id"], "examples": [e["id"] for e in ex]}
                          for r, ex in zip(rows, examples_all)] if examples_all else [])

        tok = AutoTokenizer.from_pretrained(MODEL)
        tok.padding_side = "left"
        tok.pad_token_id = tok.pad_token_id or tok.eos_token_id
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16,
                                                    attn_implementation="sdpa").cuda().eval()
        if args.adapter:
            from peft import PeftModel
            # DPO adapter 训练时叠在已合并 SFT 的权重上，adapter_config 里
            # 却只记了 base=Qwen。直接挂到原始 Qwen 上等于丢掉 SFT 的全部增益
            # 直接挂到原始 Qwen 会退回零样本水平。先合并 SFT 再叠 DPO。
            if args.base_adapter and Path(args.adapter).resolve() != Path(args.base_adapter).resolve():
                model = PeftModel.from_pretrained(model, args.base_adapter).merge_and_unload()
                print(f"[load] 先合并 base adapter: {args.base_adapter}", flush=True)
            model = PeftModel.from_pretrained(model, args.adapter).merge_and_unload()
            print(f"[load] adapter merged: {args.adapter}", flush=True)

        gen_kw = dict(max_new_tokens=args.max_new_tokens, do_sample=False,
                      pad_token_id=tok.pad_token_id)
        if args.decode == "beam4":
            gen_kw.update(num_beams=4, num_return_sequences=1)

        t_start = time.perf_counter()
        results, total_gen = [], 0
        for start in range(0, len(rows), args.batch):
            chunk = rows[start:start + args.batch]
            examples_per_row = examples_all[start:start + len(chunk)] if examples_all else None
            texts = [tok.apply_chat_template(build_messages(r["src"], exs if exs else None,
                                                              args.rag_format),
                                             tokenize=False, add_generation_prompt=True)
                     for r, exs in zip(chunk, examples_per_row or [None] * len(chunk))]
            enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = model.generate(**enc, **gen_kw)
            torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            gen = out[:, enc["input_ids"].shape[1]:]
            # 批量生成按本批最长 padding，逐句长度要数非 pad token
            gen_len = (gen != tok.pad_token_id).sum(dim=1)
            total_gen += int(gen_len.sum())
            preds = tok.batch_decode(gen, skip_special_tokens=True)
            for r, p, gl in zip(chunk, preds, gen_len):
                results.append({"id": r["id"], "domain": r["domain"], "src": r["src"],
                                "tgt": r.get("tgt", ""), "pred": p.strip(),
                                "prompt_tokens": int(enc["input_ids"].shape[1]),
                                "gen_tokens": int(gl),
                                "batch_ms_per_sentence": round(1000 * dt / len(chunk), 2)})
            if start % (args.batch * 20) == 0:
                print(f"  {start + len(chunk)}/{len(rows)}  {dt:.2f}s/batch", flush=True)

        wall = time.perf_counter() - t_start
        with (out_dir / "pass1.jsonl").open("w", encoding="utf-8", newline="\n") as fh:
            for r in results:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
        if retrieved_log:
            with (out_dir / "retrieved.jsonl").open("w", encoding="utf-8", newline="\n") as fh:
                for r in retrieved_log:
                    fh.write(json.dumps(r, ensure_ascii=False) + "\n")

        # batch=1 逐句延迟（真实交互口径）
        lat = []
        sample = rows[:args.latency_sample] if args.latency_sample else []
        if sample:
            for r in sample:
                text = tok.apply_chat_template(build_messages(r["src"], None), tokenize=False,
                                               add_generation_prompt=True)
                enc = tok([text], return_tensors="pt").to("cuda")
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                with torch.inference_mode():
                    model.generate(**enc, **gen_kw)
                torch.cuda.synchronize()
                lat.append((time.perf_counter() - t0) * 1000)
            lat.sort()
        lat_stats = {"n": len(lat),
                     "p50_ms": round(lat[len(lat) // 2], 1) if lat else None,
                     "p95_ms": round(lat[int(len(lat) * 0.95)], 1) if lat else None,
                     "mean_ms": round(sum(lat) / len(lat), 1) if lat else None}

        meta = {"run_id": args.run_id, "mode": args.mode, "split": args.split,
                "domains": args.domains, "adapter": args.adapter,
                "k": args.k if examples_all else None,
                "rag_format": args.rag_format,
                "pool": args.pool if examples_all else None, "decode": args.decode,
                "batch": args.batch, "max_new_tokens": args.max_new_tokens,
                "rows": len(results), "wallclock_min": round(wall / 60, 2),
                "sent_per_sec": round(len(results) / wall, 2),
                "gen_tokens_total": total_gen,
                "tok_per_sec": round(total_gen / wall, 1),
                "latency_batch1": lat_stats,
                "peak_vram_gb": round(torch.cuda.max_memory_allocated() / GB, 3),
                "git_commit": git_commit(), "seed": 42,
                "model": MODEL, "model_revision": os.environ.get("MODEL_REV", "main")}
        (out_dir / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
        print("\n[meta] " + json.dumps(meta, ensure_ascii=False), flush=True)
        print(f"saved -> {out_dir/'pass1.jsonl'}", flush=True)

    main(args)

run_stage('infer', 'Cell I1: batch inference',
          fn=lambda: stage_infer(SimpleNamespace(**{'run_id': 'g3_sft', 'mode': 'sft', 'split': 'test', 'domains': 'mtnt,emea,clean', 'adapter': None, 'base_adapter': None, 'k': 3, 'rag_format': 'turns', 'domain_gated': None, 'pool': 'seen', 'batch': 32, 'max_new_tokens': 160, 'decode': 'greedy', 'limit': None, 'latency_sample': 200, 'set': 'test'})),
          outputs=('runs/g3_sft/pass1.jsonl',))

[skip] Cell I1: batch inference: RUN['infer'] is False


False

### Cell I2: G5 pipeline

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import re
import sys
import time
from pathlib import Path

def stage_g5(args):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "garbage_collection_threshold:0.85")

    import torch  # noqa: E402

    ROOT = PROJ
    DATA = ROOT / "data" / "processed"
    RUNS = ROOT / "runs"
    MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    GB = 1e9

    SYSTEM = ("You are a professional English-to-French translator. Translate the user's sentence "
              "into natural, accurate French. Output only the translation.")

    CRITIC_SYS = ("You are a bilingual reviewer of English-to-French medical and social-media "
                  "translations. List concrete problems, or reply with {\"issues\": []} if the "
                  "translation is faithful and idiomatic. Output JSON only: "
                  '{"issues":[{"type":"term|idiom|grammar|omission|register","fix":"..."}],'
                  '"severity":1-5}')

    REFINER_SYS = ("You are a professional English-to-French translator. Given the source, a draft "
                   "and reviewer notes, produce the final translation. Apply the fixes, keep "
                   "everything that was already correct. Output the translation only.")

    SET_FILES = {"test": {"mtnt": "mtnt.test.jsonl", "emea": "emea.test.jsonl",
                          "clean": "clean.test.jsonl"},
                 "dev": {"mtnt": "mtnt.dev.jsonl", "emea": "emea.dev.jsonl",
                         "clean": "clean.dev.jsonl"}}


    def load(name: str) -> list[dict]:
        with (DATA / name).open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def load_glossary() -> list[dict]:
        path = DATA / "glossary.emea.json"
        if not path.exists():
            return []
        terms = json.loads(path.read_text(encoding="utf-8"))
        for t in terms:
            t["_pat"] = re.compile(r"\b" + re.escape(t["src"].lower()) + r"\b")
        return terms


    def load_retrieval(pool: str, k: int) -> dict[str, list[dict]]:
        # 索引在 data/index，不在 data/processed/index 下
        path = ROOT / "data" / "index" / "cache" / f"{pool}.top{max(k, 12)}.jsonl"
        out: dict[str, list[dict]] = {}
        if path.exists():
            with path.open(encoding="utf-8") as fh:
                for line in fh:
                    if line.strip():
                        d = json.loads(line)
                        out[d["id"]] = d["examples"][:k]
        return out


    def load_pool_rows() -> list[dict]:
        rows = []
        for name in ("mtnt.train.jsonl", "emea.train.jsonl"):
            rows += load(name)
        return rows


    def random_examples(pool: list[dict], k: int, target_tokens: int,
                        rng) -> list[dict]:
        """长度匹配的随机示例：用来做 `--rag-control random` 对照组。

        检索组与随机组放进 prompt 的 token 数接近，才能把"检索内容有效"和
        "prompt 变长本身有效"分开——否则 RAG 的增益说不清是哪来的。
        """
        lo, hi = int(target_tokens * 0.75), int(target_tokens * 1.25)
        for _ in range(200):
            picked = rng.sample(pool, k)
            total = sum(len((e["src"] + " " + e["tgt"]).split()) for e in picked)
            if lo <= total <= hi:
                return [{"src": e["src"], "tgt": e["tgt"]} for e in picked]
        return [{"src": e["src"], "tgt": e["tgt"]} for e in rng.sample(pool, k)]


    def chat(tok, system: str, user: str) -> str:
        return tok.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)


    def repeat_ratio(text: str, n: int = 4) -> float:
        toks = text.split()
        if len(toks) < n + 1:
            return 0.0
        grams = [" ".join(toks[i:i + n]) for i in range(len(toks) - n + 1)]
        return 1.0 - len(set(grams)) / len(grams)


    def guard(draft: str, final: str) -> tuple[str, str | None]:
        """任一条不满足就回退 draft，并返回回退原因。"""
        if not final.strip():
            return draft, "empty"
        if not 0.5 <= len(final.split()) / max(1, len(draft.split())) <= 2.0:
            return draft, "length"
        if repeat_ratio(final) > 0.30:
            return draft, "repeat"
        if re.search(r"[A-Za-z]{4,}\s+[A-Za-z]{4,}\s*$", final) and final.strip().endswith((" the", " and", " of")):
            return draft, "dangling"
        return final, None


    def main(args):

        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer

        torch.cuda.set_per_process_memory_fraction(0.95)
        out_dir = RUNS / args.run_id
        out_dir.mkdir(parents=True, exist_ok=True)

        tok = AutoTokenizer.from_pretrained(MODEL)
        tok.padding_side = "left"
        tok.pad_token_id = tok.pad_token_id or tok.eos_token_id

        base = AutoModelForCausalLM.from_pretrained(
            MODEL, dtype=torch.bfloat16, attn_implementation="sdpa")
        sft = PeftModel.from_pretrained(base, args.sft_adapter).merge_and_unload().cuda().eval()
        print(f"[load] SFT adapter 合并完成", flush=True)

        dpo = None
        if args.use_dpo and Path(args.dpo_adapter).exists():
            # DPO adapter 挂在已合并 SFT 的权重上：adapter_config 只记 base=Qwen，
            # 直接挂到原始 Qwen 会丢掉 SFT 的增益。deepcopy 是为了不污染轻通道的权重。
            import copy
            dpo = copy.deepcopy(sft)
            dpo = PeftModel.from_pretrained(dpo, args.dpo_adapter).merge_and_unload().eval()
            print(f"[load] DPO adapter 叠在 SFT 之上合并完成", flush=True)

        terms = load_glossary() if args.use_con else []
        retr = load_retrieval(args.pool, args.k) if args.use_rag else {}
        pool_rows = load_pool_rows() if args.rag_control == "random" else []
        rng_ctrl = __import__("random").Random(42)
        print(f"[load] 术语 {len(terms)} 条，检索缓存 {len(retr)} 条", flush=True)

        files = [SET_FILES[args.split][d] for d in args.domains.split(",") if d in SET_FILES[args.split]]
        rows = [r for f in files for r in load(f)]
        if args.limit:
            rows = rows[: args.limit]
        print(f"[data] {args.split} {len(rows)} 句", flush=True)

        def gen(model, prompts: list[str], max_new: int) -> tuple[list[str], float]:
            enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            torch.cuda.synchronize()
            dt = time.perf_counter() - t0
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            return [t.strip() for t in texts], dt / len(prompts)

        stats = {"rows": len(rows), "fallback": 0, "fallback_reasons": {}, "glossary_hits": 0,
                 "t_draft": 0.0, "t_heavy": 0.0, "t_critic": 0.0, "t_refiner": 0.0}
        results, drafts = [], []
        t_all = time.perf_counter()

        for start in range(0, len(rows), args.batch):
            chunk = rows[start:start + args.batch]

            # 1) light path：SFT 直出
            draft_prompts = [chat(tok, SYSTEM, r["src"]) for r in chunk]
            drafts_txt, t_d = gen(sft, draft_prompts, args.max_new_tokens)
            stats["t_draft"] += t_d * len(chunk)

            # 2) heavy path 的上下文：检索示例（user/assistant 轮次）+ 术语提示（最后一条 user）
            #    单条 user 消息塞示例（inline）比 user/assistant 轮次低约 0.09 COMET，
            #    显式轮次结构（turns）明显更好，所以这里按 turns 组 prompt。
            heavy_prompts, hits = [], []
            for r, d in zip(chunk, drafts_txt):
                exs = retr.get(r["id"], []) if retr else []
                if args.rag_control == "random":
                    tgt_tokens = sum(len((e["src"] + " " + e["tgt"]).split()) for e in exs) or 60
                    exs = random_examples(pool_rows, args.k, tgt_tokens, rng_ctrl)
                gl = [t for t in terms if t["_pat"].search(r["src"].lower())] if terms else []
                msgs = [{"role": "system", "content": SYSTEM}]
                for e in exs:
                    msgs.append({"role": "user", "content": e["src"]})
                    msgs.append({"role": "assistant", "content": e["tgt"]})
                tail = []
                if gl:
                    tail.append("Required terminology: " +
                                "; ".join(f"{t['src']} -> {t['fr']}" for t in gl))
                tail.append(r["src"])
                msgs.append({"role": "user", "content": "\n\n".join(tail)})
                heavy_prompts.append(tok.apply_chat_template(msgs, tokenize=False,
                                                             add_generation_prompt=True))
                hits.append(len(gl))

            heavy_model = dpo or sft
            heavy_txt, t_h = gen(heavy_model, heavy_prompts, args.max_new_tokens)
            stats["t_heavy"] += t_h * len(chunk)
            stats["glossary_hits"] += sum(hits)

            # 3) Critic
            final_txt = list(heavy_txt)
            if args.use_agent:
                critic_prompts = [chat(tok, CRITIC_SYS,
                                       f"Source: {r['src']}\nDraft: {h}\nReview JSON:")
                                  for r, h in zip(chunk, heavy_txt)]
                critiques, t_c = gen(sft, critic_prompts, 160)
                stats["t_critic"] += t_c * len(chunk)
                refiner_prompts = [chat(tok, REFINER_SYS,
                                        f"Source: {r['src']}\nDraft: {h}\nReviewer notes: {c}\n"
                                        f"Final French translation:")
                                   for r, h, c in zip(chunk, heavy_txt, critiques)]
                refined, t_r = gen(heavy_model, refiner_prompts, args.max_new_tokens)
                stats["t_refiner"] += t_r * len(chunk)
                final_txt = []
                for r, h, f in zip(chunk, heavy_txt, refined):
                    kept, reason = guard(h, f)
                    if reason:
                        stats["fallback"] += 1
                        stats["fallback_reasons"][reason] = stats["fallback_reasons"].get(reason, 0) + 1
                    final_txt.append(kept)
            else:
                critiques = [""] * len(chunk)

            for r, d, h, c, f in zip(chunk, drafts_txt, heavy_txt, critiques, final_txt):
                results.append({"id": r["id"], "domain": r["domain"], "src": r["src"],
                                "tgt": r.get("tgt", ""), "pred": f, "draft": d,
                                "heavy": h, "critic": c[:400],
                                "glossary_hits": len([t for t in terms
                                                      if t["_pat"].search(r["src"].lower())])})
                drafts.append({"id": r["id"], "domain": r["domain"], "src": r["src"],
                               "tgt": r.get("tgt", ""), "pred": d})
            if start % (args.batch * 10) == 0:
                done = start + len(chunk)
                print(f"  {done}/{len(rows)}  {time.perf_counter()-t_all:.0f}s  "
                      f"fallback {stats['fallback']}", flush=True)

        wall = time.perf_counter() - t_all
        for name, data in (("pass1", results), ("draft", drafts)):
            with (out_dir / f"{name}.jsonl").open("w", encoding="utf-8", newline="\n") as fh:
                for r in data:
                    fh.write(json.dumps(r, ensure_ascii=False) + "\n")

        meta = {"run_id": args.run_id, "split": args.split, "rows": len(rows),
                "flags": {"dpo": args.use_dpo and dpo is not None, "rag": args.use_rag,
                          "rag_control": args.rag_control,
                          "con": args.use_con, "agent": args.use_agent},
                "k": args.k, "pool": args.pool,
                "wallclock_min": round(wall / 60, 2),
                "mean_ms": {k: round(1000 * v / len(rows), 1)
                            for k, v in stats.items() if k.startswith("t_")},
                "fallback": stats["fallback"], "fallback_reasons": stats["fallback_reasons"],
                "glossary_hits": stats["glossary_hits"],
                "peak_vram_gb": round(torch.cuda.max_memory_allocated() / GB, 3)}
        (out_dir / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2),
                                           encoding="utf-8")
        print("[meta] " + json.dumps(meta, ensure_ascii=False), flush=True)

    main(args)

run_stage('g5', 'Cell I2: G5 pipeline',
          fn=lambda: stage_g5(SimpleNamespace(**{'run_id': 'g5_full', 'split': 'test', 'domains': 'mtnt,emea,clean', 'sft_adapter': None, 'dpo_adapter': None, 'k': 3, 'pool': 'seen', 'batch': 16, 'max_new_tokens': 160, 'use_dpo': None, 'use_rag': None, 'use_con': None, 'use_agent': None, 'rag_control': 'none', 'limit': None})),
          outputs=('runs/g5_full/draft.jsonl',))

[skip] Cell I2: G5 pipeline: RUN['g5'] is False


False

### Cell E1: COMET scoring

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import os
import sys
import time
from pathlib import Path

def stage_score(args):
    ROOT = PROJ
    RUNS = ROOT / "runs"

    DA_MODEL = "Unbabel/wmt22-comet-da"
    QE_LOCAL = str(ROOT / "runs" / "qe_local" / "wmt22-cometkiwi-da-mirror" / "checkpoints" / "model.ckpt")
    QE_CANDIDATES = [QE_LOCAL, "Unbabel/wmt22-cometkiwi-da"]


    def load_model(repo: str):
        """repo 可以是 HF 仓库名，也可以是本地 ckpt 路径（镜像转换产物）。"""
        from comet import download_model, load_from_checkpoint
        path = repo if Path(repo).exists() else download_model(repo)
        return load_from_checkpoint(path)


    def predict(model, data: list[dict], batch_size: int):
        """兼容 comet 2.x 的两种 predict 签名与两种返回类型。"""
        try:
            out = model.predict(data, batch_size=batch_size, gpus=1, progress_bar=False)
        except TypeError:
            out = model.predict(data, batch_size=batch_size, accelerator="gpu", devices=1,
                                progress_bar=False)
        if isinstance(out, tuple):          # (scores, system_score)
            return list(out[0]), float(out[1])
        scores = getattr(out, "scores", None) or out["scores"]
        system = getattr(out, "system_score", None)
        if system is None:
            system = out.get("system_score")
        return list(scores), float(system)


    def read_pass1(run_id: str, variant: str = "pass1") -> list[dict]:
        path = RUNS / run_id / f"{variant}.jsonl"
        with path.open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def main(args):

        import torch
        print(f"[env] torch={torch.__version__} cuda={torch.cuda.is_available()}", flush=True)

        da = load_model(DA_MODEL)
        print(f"[load] {DA_MODEL} ok", flush=True)

        qe = None
        qe_used = None
        if not args.skip_qe:
            for repo in ([args.qe_model] if args.qe_model else QE_CANDIDATES):
                try:
                    qe = load_model(repo)
                    qe_used = repo
                    print(f"[load] QE = {repo} ok", flush=True)
                    break
                except Exception as exc:  # noqa: BLE001
                    print(f"[warn] QE {repo} 不可用：{type(exc).__name__} {str(exc)[:90]}", flush=True)
            if qe is None:
                print("[warn] 无可用 QE 模型，Kiwi 分数留空（走 FEASIBILITY_REVIEW §4 降级）", flush=True)

        for run_id in [r.strip() for r in args.runs.split(",") if r.strip()]:
            rows = read_pass1(run_id, args.variant)
            if not rows:
                print(f"[skip] {run_id} 无数据", flush=True)
                continue
            da_data = [{"src": r["src"], "mt": r["pred"], "ref": r["tgt"]} for r in rows]
            t0 = time.perf_counter()
            da_seg, da_sys = predict(da, da_data, args.batch_size)
            da_time = time.perf_counter() - t0

            qe_seg = [None] * len(rows)
            qe_time = 0.0
            if qe is not None:
                qe_data = [{"src": r["src"], "mt": r["pred"]} for r in rows]
                t1 = time.perf_counter()
                qe_seg, _ = predict(qe, qe_data, args.batch_size)
                qe_time = time.perf_counter() - t1

            suffix = "" if args.variant == "pass1" else f"_{args.variant}"
            out = RUNS / run_id / f"scores{suffix}.jsonl"
            with out.open("w", encoding="utf-8", newline="\n") as fh:
                for r, d, q in zip(rows, da_seg, qe_seg):
                    rec = {"id": r["id"], "domain": r["domain"], "src": r["src"], "tgt": r["tgt"],
                           "pred": r["pred"], "comet_da": round(float(d), 5),
                           "kiwi": (round(float(q), 5) if q is not None else None),
                           "src_tokens": r.get("prompt_tokens"), "gen_tokens": r.get("gen_tokens")}
                    fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

            by_domain: dict[str, list[float]] = {}
            for r, d in zip(rows, da_seg):
                by_domain.setdefault(r["domain"], []).append(float(d))
            summary = {"run_id": run_id, "rows": len(rows),
                       "comet_da_system": round(da_sys, 5),
                       "comet_da_by_domain": {k: round(sum(v) / len(v), 5) for k, v in by_domain.items()},
                       "macro_comet_da": round(sum(sum(v) / len(v) for v in by_domain.values())
                                               / len(by_domain), 5),
                       "da_seconds": round(da_time, 1), "qe_seconds": round(qe_time, 1),
                       "sent_per_sec": round(len(rows) / max(da_time, 1e-6), 1),
                       "qe_model": qe_used}
            (RUNS / run_id / f"scores_summary{suffix}.json").write_text(
                json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"[{run_id}] " + json.dumps(summary, ensure_ascii=False), flush=True)

    main(args)

run_stage('score', 'Cell E1: COMET scoring',
          fn=lambda: stage_score(SimpleNamespace(**{'runs': 'g3_sft', 'batch_size': 64, 'variant': 'pass1', 'skip_qe': None, 'qe_model': None})),
          outputs=('runs/g3_sft/scores.jsonl',),
          comet=('src/eval/score_comet.py', {'runs': 'g3_sft', 'batch_size': 64, 'variant': 'pass1', 'skip_qe': None, 'qe_model': None}))

[skip] Cell E1: COMET scoring: RUN['score'] is False


False

### Cell E2: metrics

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import re
import statistics
import sys
from pathlib import Path

def stage_metrics(args):
    ROOT = PROJ
    RUNS = ROOT / "runs"
    DATA = ROOT / "data" / "processed"
    REPORTS = ROOT / "reports"
    CONCLUSION_DOMAINS = ("mtnt", "emea")
    WORD = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿŒœÆæ0-9][A-Za-zÀ-ÖØ-öø-ÿŒœÆæ0-9'\-]*")


    def load_jsonl(path: Path) -> list[dict]:
        with path.open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def load_glossary() -> dict[str, list[dict]]:
        """两个域各一张表：EMEA 术语表 + MTNT 俚语表。

        MTNT 表里 mode=omit 的条目没有固定对应（整句改写），不参与命中率，
        否则会把"本来就没有固定译法"的条目算成 miss，压低指标。
        """
        out: dict[str, list[dict]] = {"emea": [], "mtnt": []}

        emea = DATA / "glossary.emea.json"
        if emea.exists():
            terms = json.loads(emea.read_text(encoding="utf-8"))
            for t in terms:
                t["_src_pat"] = re.compile(r"\b" + re.escape(t["src"].lower()) + r"\b")
                acc = [a.lower() for a in t.get("accepted", []) if a]
                t["_acc_pats"] = [re.compile(r"\b" + re.escape(a) + r"\b") for a in acc]
            out["emea"] = terms

        mtnt = DATA / "glossary.mtnt.json"
        if mtnt.exists():
            terms = [t for t in json.loads(mtnt.read_text(encoding="utf-8"))
                     if t.get("mode") != "omit"]
            for t in terms:
                t["_src_pat"] = re.compile(r"\b" + re.escape(t["src"].lower()) + r"\b")
                acc = [a.lower() for a in t.get("accepted", []) if a]
                t["_acc_pats"] = [re.compile(r"\b" + re.escape(a) + r"\b") for a in acc]
            out["mtnt"] = terms
        return out


    def ngram_repeat_ratio(text: str, n: int = 4) -> float:
        toks = text.split()
        if len(toks) < n + 1:
            return 0.0
        grams = [" ".join(toks[i:i + n]) for i in range(len(toks) - n + 1)]
        return 1.0 - len(set(grams)) / len(grams)


    def domain_metrics(rows: list[dict], terms: list[dict]) -> dict:
        from sacrebleu.metrics import BLEU, CHRF

        preds = [r["pred"] for r in rows]
        refs = [r["tgt"] for r in rows]
        out = {"rows": len(rows),
               "comet_da": round(statistics.mean(r["comet_da"] for r in rows), 4),
               "bleu": round(BLEU(tokenize="13a").corpus_score(preds, [refs]).score, 2),
               "chrfpp": round(CHRF(char_order=6, word_order=2).corpus_score(preds, [refs]).score, 2)}

        kiwi = [r["kiwi"] for r in rows if r.get("kiwi") is not None]
        out["kiwi"] = round(statistics.mean(kiwi), 4) if kiwi else None

        # 重复退化
        out["repetition_rate"] = round(
            sum(1 for p in preds if ngram_repeat_ratio(p) > 0.30) / max(1, len(preds)), 4)
        bad = 0
        for r in rows:
            p = r["pred"].strip()
            if not p:
                bad += 1
                continue
            ratio = len(p.split()) / max(1, len(r["src"].split()))
            if not 0.3 <= ratio <= 3.0 or p.lower() == r["src"].strip().lower():
                bad += 1
        out["degeneration_rate"] = round(bad / max(1, len(rows)), 4)
        out["length_ratio"] = round(
            statistics.mean(len(r["pred"].split()) / max(1, len(r["src"].split())) for r in rows), 3)

        # 术语命中（只在源句含术语的子集上算）
        if terms:
            opp = hit = 0
            for r in rows:
                src = r["src"].lower()
                for t in terms:
                    if t["_src_pat"].search(src):
                        opp += 1
                        if any(p.search(r["pred"].lower()) for p in t["_acc_pats"]):
                            hit += 1
            out["term_opportunities"] = opp
            out["term_hit_rate"] = round(hit / opp, 4) if opp else None
        return out


    def run_metrics(run_id: str, terms: list[dict]) -> dict | None:
        scores = RUNS / run_id / "scores.jsonl"
        if not scores.exists():
            return None
        rows = load_jsonl(scores)
        by_dom: dict[str, list[dict]] = {}
        for r in rows:
            by_dom.setdefault(r["domain"], []).append(r)

        res = {"run_id": run_id,
               "by_domain": {d: domain_metrics(v, terms.get(d, [])) for d, v in by_dom.items()}}
        conclude = [res["by_domain"][d]["comet_da"] for d in CONCLUSION_DOMAINS if d in res["by_domain"]]
        res["macro_comet_da"] = round(statistics.mean(conclude), 4) if conclude else None

        meta = RUNS / run_id / "meta.json"
        if meta.exists():
            m = json.loads(meta.read_text(encoding="utf-8"))
            res["latency_p50_ms"] = m.get("latency_batch1", {}).get("p50_ms")
            res["latency_p95_ms"] = m.get("latency_batch1", {}).get("p95_ms")
            res["peak_vram_gb"] = m.get("peak_vram_gb")
            res["sent_per_sec"] = m.get("sent_per_sec")
            res["decode"] = m.get("decode")
            res["k"] = m.get("k")
        return res


    def main(args):

        terms = load_glossary()  # {domain: [terms]}
        if args.all or not args.runs:
            ids = [p.name for p in RUNS.iterdir()
                   if p.is_dir() and (p / "scores.jsonl").exists() and not p.name.startswith("_")]
        else:
            ids = [r.strip() for r in args.runs.split(",") if r.strip()]

        out = {}
        for rid in ids:
            res = run_metrics(rid, terms)
            if res is None:
                print(f"[skip] {rid} 无 scores.jsonl", flush=True)
                continue
            out[rid] = res
            line = (f"{rid:22s} macro={res['macro_comet_da']}  "
                    + "  ".join(f"{d}={v['comet_da']:.4f}" for d, v in res["by_domain"].items()))
            print(line, flush=True)

        REPORTS.mkdir(parents=True, exist_ok=True)
        prev_path = REPORTS / "metrics.json"
        merged = json.loads(prev_path.read_text(encoding="utf-8")) if prev_path.exists() else {}
        merged.update(out)
        prev_path.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n-> {prev_path.relative_to(ROOT).as_posix()}  ({len(merged)} runs)")

    main(args)

run_stage(None, 'Cell E2: metrics',
          fn=lambda: stage_metrics(SimpleNamespace(**{'runs': None, 'all': True})),
          outputs=(), always=True)


=== 14:30:08  Cell E2: metrics ===


g1_zs                  macro=0.7471  mtnt=0.7292  emea=0.7651  clean=0.9004


g1_zs_dev              macro=0.7593  mtnt=0.7274  emea=0.7912  clean=0.8907


g2_rag                 macro=0.7633  mtnt=0.7264  emea=0.8002  clean=0.9503


g2_rag_k1_dev          macro=0.7727  mtnt=0.7274  emea=0.8180  clean=0.9280


g2_rag_k3_dev          macro=0.7847  mtnt=0.7298  emea=0.8396  clean=0.9441


... (16 outputs left out)

g5_no_rag              macro=0.761  mtnt=0.7338  emea=0.7881  clean=0.8847


g5_no_rag_len          macro=0.7538  mtnt=0.7300  emea=0.7776  clean=0.8851



-> reports/metrics.json  (23 runs)
[done] Cell E2: metrics in 0.4 min (14:30:30)


True

Main results on the 3,271-sentence test set. COMET-DA per domain and the two-domain macro average (MTNT+EMEA, equal weight); BLEU/chrF++/term hit reported on EMEA; latency is batch-1 p50; VRAM is peak allocated.

| System | MTNT | EMEA | Macro | BLEU | chrF++ | Term % | p50 ms | VRAM GB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| G1 Zero-shot | 0.7292 | 0.7651 | \textbf{0.7471} | 22.9 | 48.9 | 35.1 | 559 | 3.66 |
| G2 Zero-shot + RAG | 0.7264 | 0.8002 | \textbf{0.7633} | 28.8 | 54.0 | 56.7 | 526 | 6.93 |
| G3 LoRA SFT | 0.7691 | 0.8271 | \textbf{0.7981} | 39.6 | 61.9 | 70.2 | 555 | 3.66 |
| G5* SFT + DPO | 0.7736 | 0.8294 | \textbf{0.8015} | 40.8 | 62.6 | 72.5 | 722 | 3.66 |
| G4 SFT + RAG | 0.7544 | 0.8110 | \textbf{0.7827} | 35.6 | 58.4 | 66.8 | 544 | 5.80 |
| G4 (inline context) | 0.6688 | 0.7780 | \textbf{0.7234} | 32.8 | 55.8 | 65.1 | 721 | 5.79 |
| G4 (domain-gated) | 0.7541 | 0.8106 | \textbf{0.7823} | 35.7 | 58.4 | 66.8 | 722 | 5.80 |
| SFT + augmented data | 0.7391 | 0.7645 | \textbf{0.7518} | 32.3 | 55.2 | 56.9 | 789 | 3.66 |
| G5 full pipeline | 0.7298 | 0.7807 | \textbf{0.7552} | 33.8 | 58.4 | 75.8 | -- | 7.54 |

The gains fall in a clear order. First, fine-tuning dominates: +0.051 macro over zero-shot, and
the gain is larger on medical text (+0.062) than on social media (+0.040), which is what one
expects from a model that has to learn a formal register it does not natively produce.

Second, retrieval helps only before fine-tuning. On zero-shot it lifts EMEA by 0.035 and leaves
MTNT unchanged. On the fine-tuned model it costs 0.015 on both domains. The same pool, the same
prompt format, opposite sign: the variable is the model state, not the retrieval index. Two
controls rule out the obvious explanations. Domain-gated retrieval, which removes the 11.8% of
MTNT queries that retrieve EMEA boilerplate, changes the macro by 0.0004. And retrieval beats a
length-matched random context by only 0.002 (p = 0.33), so the pool carries little usable signal
in this setting.

Third, preference optimisation is measurable but immaterial: +0.003 (CI +0.0015 to +0.0046,
p < 1e-4) is statistically detectable yet a quarter of the minimum detectable effect.

The context format matters more than the context content. Injecting the same three examples as
extra `user`/`assistant` turns costs 0.015; stuffing them into a single user message costs 0.075.
A model fine-tuned on single-turn data does not accept multi-turn demonstrations gracefully, and
compressing them into one message is worse still.

Paired bootstrap (10,000 resamples, comment-id clusters for MTNT) with Holm correction within each family. Delta is macro COMET-DA of the first system minus the second.

| Comparison | Delta | 95% CI | p | Sig. |
| --- | --- | --- | --- | --- |
| g5_full $-$ g3_sft | -0.0440 | [-0.0478, -0.0401] | 0.0000 | yes |
| g5_full $-$ g5_no_dpo | +0.0027 | [+0.0001, +0.0052] | 0.0412 | no |
| g5_full $-$ g5_no_rag | -0.0062 | [-0.0100, -0.0026] | 0.0012 | yes |
| g5_full $-$ g5_no_rag_len | +0.0020 | [-0.0020, +0.0061] | 0.3328 | no |
| g5_full $-$ g5_no_con | -0.0046 | [-0.0067, -0.0026] | 0.0000 | yes |
| g5_full $-$ g5_no_agent | -0.0238 | [-0.0267, -0.0209] | 0.0000 | yes |
| g5_dpo_direct $-$ g3_sft | +0.0031 | [+0.0015, +0.0046] | 0.0000 | yes |
| g3_sft_aug $-$ g3_sft | -0.0516 | [-0.0560, -0.0472] | 0.0000 | yes |

## 6. Ablations

### Cell E3: paired bootstrap

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import random
import statistics
import sys
from pathlib import Path

def stage_bootstrap(args):
    ROOT = PROJ
    RUNS = ROOT / "runs"
    DATA = ROOT / "data" / "processed"
    REPORTS = ROOT / "reports"
    CONCLUSION = ("mtnt", "emea")

    FAMILIES = {
        "main": [("g3_sft", "g1_zs"), ("g4_sft_rag", "g3_sft"),
                 ("g5_full", "g4_sft_rag"), ("g5_full", "g1_zs")],
        "ablation": [("g5_full", "g5_no_dpo"), ("g5_full", "g5_no_rag"),
                     ("g5_full", "g5_no_con"), ("g5_full", "g5_no_agent")],
        # 本次跑完矩阵后实际需要的比较族：G5 整体 vs 轻通道，以及四个模块的留一
        "g5": [("g5_full", "g3_sft"), ("g5_full", "g5_no_dpo"), ("g5_full", "g5_no_rag"),
               ("g5_full", "g5_no_rag_len"), ("g5_full", "g5_no_con"),
               ("g5_full", "g5_no_agent")],
    }


    def load_scores(run: str) -> dict[str, dict]:
        path = RUNS / run / "scores.jsonl"
        with path.open(encoding="utf-8") as fh:
            return {json.loads(l)["id"]: json.loads(l) for l in fh if l.strip()}


    def comment_ids(domain: str) -> dict[str, str]:
        """MTNT 的 comment_id 分组键；EMEA 用句子自身（iid）。"""
        path = DATA / f"{domain}.test.jsonl"
        if not path.exists():
            return {}
        out = {}
        with path.open(encoding="utf-8") as fh:
            for l in fh:
                if l.strip():
                    d = json.loads(l)
                    out[d["id"]] = d.get("comment_id") or d["id"]
        return out


    def paired(a: dict, b: dict, metric: str, iterations: int, seed: int = 42) -> dict:
        rng = random.Random(seed)
        groups: dict[str, list[str]] = {}
        for dom in CONCLUSION:
            ids = [i for i in a if i in b and a[i]["domain"] == dom]
            if not ids:
                continue
            cid = comment_ids(dom)
            key = (lambda i: cid.get(i, i)) if dom == "mtnt" else (lambda i: i)
            for i in ids:
                groups.setdefault(f"{dom}|{key(i)}", []).append(i)
        keys = list(groups)

        point = statistics.mean(a[i][metric] - b[i][metric]
                               for g in groups.values() for i in g)

        diffs = []
        for _ in range(iterations):
            sample = [g for _ in keys for g in groups[rng.choice(keys)]]
            if not sample:
                continue
            diffs.append(statistics.mean(a[i][metric] - b[i][metric] for i in sample))
        diffs.sort()
        lo = diffs[int(0.025 * len(diffs))]
        hi = diffs[int(0.975 * len(diffs)) - 1]
        p = 2 * min(sum(1 for d in diffs if d <= 0) / len(diffs),
                    sum(1 for d in diffs if d >= 0) / len(diffs))
        return {"metric": metric, "delta": round(point, 4), "ci_lo": round(lo, 4),
                "ci_hi": round(hi, 4), "p_value": round(min(1.0, p), 4),
                "clusters": len(keys), "iterations": len(diffs)}


    def corpus_bleu_delta(a: dict, b: dict, metric: str, iterations: int, seed: int = 42) -> dict:
        from sacrebleu.metrics import BLEU, CHRF

        rng = random.Random(seed)
        ids = [i for i in a if i in b]
        by_dom = {d: [i for i in ids if a[i]["domain"] == d] for d in CONCLUSION}
        ids = [i for d in CONCLUSION for i in by_dom[d]]

        def make():
            return BLEU(tokenize="13a") if metric == "bleu" else CHRF(char_order=6, word_order=2)

        def score(sub):
            vals_a, vals_b = [], []
            for d in CONCLUSION:
                sel = [i for i in sub if a[i]["domain"] == d]
                if not sel:
                    continue
                refs = [[a[i]["tgt"] for i in sel]]
                vals_a.append(make().corpus_score([a[i]["pred"] for i in sel], refs).score)
                vals_b.append(make().corpus_score([b[i]["pred"] for i in sel], refs).score)
            return statistics.mean(vals_a) - statistics.mean(vals_b)

        point = score(ids)
        diffs = sorted(score([ids[rng.randrange(len(ids))] for _ in range(len(ids))])
                       for _ in range(iterations))
        lo = diffs[int(0.025 * len(diffs))]
        hi = diffs[int(0.975 * len(diffs)) - 1]
        p = 2 * min(sum(1 for d in diffs if d <= 0) / len(diffs),
                    sum(1 for d in diffs if d >= 0) / len(diffs))
        return {"metric": metric, "delta": round(point, 2), "ci_lo": round(lo, 2),
                "ci_hi": round(hi, 2), "p_value": round(min(1.0, p), 4),
                "rows": len(ids), "iterations": len(diffs)}


    def holm(results: list[dict], alpha: float = 0.05) -> None:
        ordered = sorted(results, key=lambda r: r["p_value"])
        m = len(ordered)
        for i, r in enumerate(ordered):
            r["holm_threshold"] = round(alpha / (m - i), 4)
            r["significant"] = r["p_value"] <= alpha / (m - i)
        for r in results:
            r.setdefault("holm_threshold", None)
            r.setdefault("significant", False)


    def main(args):

        pairs = FAMILIES[args.family] if args.family else [(args.a, args.b)]
        out = []
        for a_id, b_id in pairs:
            if not (RUNS / a_id / "scores.jsonl").exists() or not (RUNS / b_id / "scores.jsonl").exists():
                print(f"[skip] {a_id} vs {b_id}：缺 scores.jsonl", flush=True)
                continue
            a, b = load_scores(a_id), load_scores(b_id)
            if args.metric in ("bleu", "chrfpp"):
                m = "bleu" if args.metric == "bleu" else "chrf"
                res = corpus_bleu_delta(a, b, m, max(200, args.iterations // 10))
            else:
                res = paired(a, b, args.metric, args.iterations)
            res.update({"a": a_id, "b": b_id})
            out.append(res)
            print(f"{a_id:16s} - {b_id:16s} Δ={res['delta']:+.4f} "
                  f"[{res['ci_lo']:+.4f}, {res['ci_hi']:+.4f}] p={res['p_value']:.4f}", flush=True)

        if not out:
            return
        holm(out)
        REPORTS.mkdir(parents=True, exist_ok=True)
        path = REPORTS / "bootstrap.json"
        prev = json.loads(path.read_text(encoding="utf-8")) if path.exists() else []
        prev = [r for r in prev if (r["a"], r["b"]) not in {(x["a"], x["b"]) for x in out}]
        path.write_text(json.dumps(prev + out, ensure_ascii=False, indent=2), encoding="utf-8")
        sig = sum(1 for r in out if r["significant"])
        print(f"\n{sig}/{len(out)} significant after Holm -> "
              f"{path.relative_to(ROOT).as_posix()}", flush=True)

    main(args)

run_stage(None, 'Cell E3: paired bootstrap',
          fn=lambda: stage_bootstrap(SimpleNamespace(**{'a': None, 'b': None, 'family': 'g5', 'metric': 'comet_da', 'iterations': 10000})),
          outputs=(), always=True)


=== 14:30:30  Cell E3: paired bootstrap ===


g5_full          - g3_sft           Δ=-0.0440 [-0.0478, -0.0401] p=0.0000


g5_full          - g5_no_dpo        Δ=+0.0027 [+0.0001, +0.0052] p=0.0412


g5_full          - g5_no_rag        Δ=-0.0062 [-0.0100, -0.0026] p=0.0012


g5_full          - g5_no_rag_len    Δ=+0.0020 [-0.0020, +0.0061] p=0.3328


    ... Cell E3: paired bootstrap: 1 min elapsed


g5_full          - g5_no_con        Δ=-0.0046 [-0.0067, -0.0026] p=0.0000


g5_full          - g5_no_agent      Δ=-0.0238 [-0.0267, -0.0209] p=0.0000



4/6 significant after Holm -> reports/bootstrap.json


[done] Cell E3: paired bootstrap in 1.3 min (14:31:49)


True

### Cell E4: routing sweep

In [ ]:
from __future__ import annotations

import argparse
import io
import json
import statistics
import sys
from pathlib import Path

def stage_route(args):
    ROOT = PROJ
    RUNS = ROOT / "runs"
    REPORTS = ROOT / "reports"
    CONCLUSION = ("mtnt", "emea")
    TAUS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]


    def load_jsonl(path: Path) -> list[dict]:
        with path.open(encoding="utf-8") as fh:
            return [json.loads(l) for l in fh if l.strip()]


    def main(args):

        run_dir = RUNS / args.run
        heavy = {r["id"]: r for r in load_jsonl(run_dir / "scores.jsonl")}
        light = {r["id"]: r for r in load_jsonl(run_dir / "scores_draft.jsonl")}
        meta = json.loads((run_dir / "meta.json").read_text(encoding="utf-8"))
        ms = meta["mean_ms"]

        ids = [i for i in heavy if i in light and light[i].get("kiwi") is not None]
        print(f"[route] {len(ids)} 句可用（heavy={len(heavy)} light={len(light)}）", flush=True)

        def macro(sel) -> dict:
            per = {}
            for dom in CONCLUSION:
                vals = [sel(i) for i in ids if heavy[i]["domain"] == dom]
                if vals:
                    per[dom] = statistics.mean(vals)
            return {"macro": round(statistics.mean(per.values()), 4) if per else None,
                    "by_domain": {k: round(v, 4) for k, v in per.items()}}

        t_light = ms.get("t_draft", 0) + ms.get("t_qe", 0)
        t_heavy_extra = (ms.get("t_heavy", 0) + ms.get("t_critic", 0) +
                         ms.get("t_refiner", 0) + ms.get("t_retr", 0))

        g4_meta_path = RUNS / args.g4 / "meta.json"
        g4_lat = None
        if g4_meta_path.exists():
            g4_lat = json.loads(g4_meta_path.read_text(encoding="utf-8")).get("latency_batch1", {}).get("p50_ms")
        budget = (g4_lat or 600) * 1.15

        points = []
        # always-light / always-heavy 两个端点
        pts = {
            "always_light": (macro(lambda i: light[i]["comet_da"]), t_light, 0.0),
            "always_heavy": (macro(lambda i: heavy[i]["comet_da"]), t_light + t_heavy_extra, 1.0),
        }
        for k, (m, lat, hr) in pts.items():
            points.append({"tau": None, "label": k, **m, "latency_ms": round(lat, 1),
                           "heavy_ratio": hr})

        for tau in TAUS:
            sel = lambda i, tau=tau: (light[i]["comet_da"] if light[i]["kiwi"] >= tau
                                      else heavy[i]["comet_da"])
            m = macro(sel)
            ratio = sum(1 for i in ids if light[i]["kiwi"] < tau) / len(ids)
            points.append({"tau": tau, **m,
                           "latency_ms": round(t_light + ratio * t_heavy_extra, 1),
                           "heavy_ratio": round(ratio, 3)})

        light_macro = pts["always_light"][0]["macro"]
        heavy_macro = pts["always_heavy"][0]["macro"]
        gain = heavy_macro - light_macro

        feasible = [p for p in points if p["tau"] is not None and p["latency_ms"] <= budget]
        relaxed = False
        if not feasible:
            relaxed = True
            feasible = [p for p in points if p["tau"] is not None and p["latency_ms"] <= (g4_lat or 600) * 1.30]
        chosen = max(feasible, key=lambda p: p["macro"]) if feasible else None

        out = {"run_id": args.run, "n_sentences": len(ids),
               "reference": {"always_light": light_macro, "always_heavy": heavy_macro,
                             "heavy_gain": round(gain, 4)},
               "latency_model": {"t_light_ms": round(t_light, 1),
                                 "t_heavy_extra_ms": round(t_heavy_extra, 1),
                                 "g4_p50_ms": g4_lat, "budget_ms": round(budget, 1),
                                 "relaxed_to_1_30": relaxed},
               "points": points, "chosen_tau": chosen}
        (run_dir / "routing.json").write_text(json.dumps(out, ensure_ascii=False, indent=2),
                                             encoding="utf-8")

        print(f"[route] always-light {light_macro:.4f}  always-heavy {heavy_macro:.4f}  "
              f"增益 {gain:+.4f}", flush=True)
        print(f"[route] 预算 {budget:.0f}ms（G4 p50={g4_lat}）", flush=True)
        for p in points:
            tag = f"τ={p['tau']}" if p["tau"] is not None else p["label"]
            print(f"  {tag:>13s}  macro={p['macro']:.4f}  lat={p['latency_ms']:6.0f}ms  "
                  f"heavy={p['heavy_ratio']:.0%}", flush=True)
        if chosen:
            if gain > 1e-6:
                share = f"（heavy 增益的 {100*(chosen['macro']-light_macro)/gain:.0f}%）"
            else:
                share = "（重路径无正增益：路由应全部走轻通道）"
            print(f"[route] τ* = {chosen['tau']}  macro={chosen['macro']:.4f} "
                  f"lat={chosen['latency_ms']:.0f}ms {share}", flush=True)
        else:
            print("[route] 无可行 τ，需放宽预算或检查 Kiwi 分布", flush=True)

    main(args)

run_stage(None, 'Cell E4: routing sweep',
          fn=lambda: stage_route(SimpleNamespace(**{'run': 'g5_full', 'g4': 'g4_sft_rag_inline'})),
          outputs=(), always=True)


=== 14:31:49  Cell E4: routing sweep ===


[route] 3271 句可用（heavy=3271 light=3271）


[route] always-light 0.7974  always-heavy 0.7553  增益 -0.0421


[route] 预算 829ms（G4 p50=721.3）


   always_light  macro=0.7974  lat=   118ms  heavy=0%


   always_heavy  macro=0.7553  lat=   492ms  heavy=100%


... (7 outputs left out)

         τ=0.85  macro=0.7792  lat=   302ms  heavy=49%


[route] τ* = 0.6  macro=0.7979 lat=141ms （重路径无正增益：路由应全部走轻通道）


[done] Cell E4: routing sweep in 0.0 min (14:31:49)


True

### Cell E5: tables

In [ ]:
from __future__ import annotations

import io
import json
import sys
from pathlib import Path

def stage_tables(args):
    ROOT = PROJ
    REPORTS = ROOT / "reports"
    TABLES = REPORTS / "tables"
    DOMAINS = ("mtnt", "emea")

    ORDER = ["g1_zs", "g2_rag", "g3_sft", "g3_sft_aug", "g4_sft_rag", "g4_sft_rag_inline",
             "g4_sft_rag_gated", "g5_dpo_direct", "g5_full", "g5_no_dpo", "g5_no_rag",
             "g5_no_rag_len", "g5_no_con", "g5_no_agent"]
    LABEL = {
        "g1_zs": "G1 零样本",
        "g2_rag": "G2 +RAG",
        "g3_sft": "G3 SFT",
        "g3_sft_aug": "G3+ 数据增强 SFT",
        "g4_sft_rag": "G4 SFT+RAG（few-shot 轮次）",
        "g4_sft_rag_inline": "G4 SFT+RAG（inline）",
        "g4_sft_rag_gated": "G4 SFT+RAG（域门控）",
        "g5_dpo_direct": "G5* SFT+DPO（无上下文）",
        "g5_full": "G5 完整管线（heavy）",
        "g5_no_dpo": "−DPO",
        "g5_no_rag": "−RAG",
        "g5_no_rag_len": "−RAG（长度匹配随机）",
        "g5_no_con": "−术语",
        "g5_no_agent": "−Agent",
    }


    def load(path: Path, default=None):
        if not path.exists():
            return default
        return json.loads(path.read_text(encoding="utf-8"))


    def fmt(v, nd=4):
        return "—" if v is None else f"{v:.{nd}f}"


    def main_table(metrics: dict) -> str:
        head = ("| 系统 | MTNT | EMEA | 两域宏平均 | BLEU(EMEA) | chrF++(EMEA) | "
                "术语命中(EMEA) | p50 延迟 | 峰值显存 |\n"
                "| --- | --- | --- | --- | --- | --- | --- | --- | --- |\n")
        rows = []
        for rid in ORDER:
            m = metrics.get(rid)
            if not m:
                continue
            bd = m["by_domain"]
            mtnt = bd.get("mtnt", {}).get("comet_da")
            emea = bd.get("emea", {}).get("comet_da")
            em = bd.get("emea", {})
            th = em.get("term_hit_rate")
            rows.append(f"| {LABEL.get(rid, rid)} | {fmt(mtnt)} | {fmt(emea)} | "
                        f"**{fmt(m.get('macro_comet_da'))}** | {fmt(em.get('bleu'), 1)} | "
                        f"{fmt(em.get('chrfpp'), 1)} | "
                        f"{'—' if th is None else f'{100*th:.1f}%'} | "
                        f"{'—' if not m.get('latency_p50_ms') else str(int(m['latency_p50_ms']))+' ms'} | "
                        f"{fmt(m.get('peak_vram_gb'), 2)} GB |")
        return head + "\n".join(rows) + "\n"


    def ablation_table(metrics: dict) -> str:
        base = metrics.get("g5_full")
        if not base:
            return "_（g5_full 未完成，暂无消融表）_\n"
        head = ("| 变体 | MTNT | EMEA | 宏平均 | Δ 宏平均 | Δ 延迟 |\n"
                "| --- | --- | --- | --- | --- | --- |\n")
        rows = []
        for rid in ["g5_full", "g5_no_dpo", "g5_no_rag", "g5_no_rag_len", "g5_no_con", "g5_no_agent"]:
            m = metrics.get(rid)
            if not m:
                continue
            d = None if rid == "g5_full" else (
                None if m.get("macro_comet_da") is None or base.get("macro_comet_da") is None
                else m["macro_comet_da"] - base["macro_comet_da"])
            lat = m.get("latency_p50_ms")
            dlat = None if (lat is None or not base.get("latency_p50_ms")) else lat - base["latency_p50_ms"]
            rows.append(f"| {LABEL.get(rid, rid)} | "
                        f"{fmt(m['by_domain'].get('mtnt', {}).get('comet_da'))} | "
                        f"{fmt(m['by_domain'].get('emea', {}).get('comet_da'))} | "
                        f"**{fmt(m.get('macro_comet_da'))}** | "
                        f"{'—' if d is None else f'{d:+.4f}'} | "
                        f"{'—' if dlat is None else f'{dlat:+.0f} ms'} |")
        return head + "\n".join(rows) + "\n"


    def routing_table() -> str:
        rt = load(ROOT / "runs" / "g5_full" / "routing.json")
        if not rt:
            return "_（routing.json 未生成）_\n"
        ref = rt["reference"]
        head = ("| 条件 | 宏平均 COMET | 延迟 | 重路径占比 |\n| --- | --- | --- | --- |\n")
        rows = [f"| always-light（=SFT 直出） | {ref['always_light']:.4f} | — | 0% |",
                f"| always-heavy | {ref['always_heavy']:.4f} | — | 100% |"]
        for p in rt["points"]:
            if p["tau"] is None:
                continue
            star = " ⭐" if rt.get("chosen_tau") and p["tau"] == rt["chosen_tau"]["tau"] else ""
            rows.append(f"| τ = {p['tau']:.2f}{star} | {p['macro']:.4f} | "
                        f"{p['latency_ms']:.0f} ms | {p['heavy_ratio']*100:.0f}% |")
        body = "\n".join(rows)
        note = (f"\n延迟模型：t_light={rt['latency_model']['t_light_ms']:.0f}ms，"
                f"重路径增量={rt['latency_model']['t_heavy_extra_ms']:.0f}ms，"
                f"预算={rt['latency_model']['budget_ms']:.0f}ms"
                f"（G4 p50 {rt['latency_model']['g4_p50_ms']}×1.15"
                f"{'，已放宽到 1.30' if rt['latency_model']['relaxed_to_1_30'] else ''}）。\n")
        return head + body + "\n" + note


    def attribution_table(metrics: dict) -> str:
        v = {}
        for rid in ("g1_zs", "g2_rag", "g3_sft", "g4_sft_rag", "g5_full"):
            m = metrics.get(rid)
            if m and m.get("macro_comet_da") is not None:
                v[rid] = m["macro_comet_da"]
        if len(v) < 4:
            return "_（主实验未跑全，暂无归因表）_\n"
        d_sft = v["g3_sft"] - v["g1_zs"]
        d_rag = v["g2_rag"] - v["g1_zs"]
        d_syn = v["g4_sft_rag"] - v["g3_sft"] - d_rag
        d_adv = v.get("g5_full", float("nan")) - v["g4_sft_rag"]
        total = v.get("g5_full", float("nan")) - v["g1_zs"]

        def pct(x):
            return "—" if total == 0 or total != total else f"{100*x/total:+.0f}%"

        head = ("| 因子 | Δ 宏平均 | 占总提升 |\n| --- | --- | --- |\n")
        rows = [f"| Δ_SFT = G3 − G1 | {d_sft:+.4f} | {pct(d_sft)} |",
                f"| Δ_RAG = G2 − G1 | {d_rag:+.4f} | {pct(d_rag)} |",
                f"| Δ_Synergy = G4 − G3 − Δ_RAG | {d_syn:+.4f} | {pct(d_syn)} |",
                f"| Δ_Advanced = G5 − G4 | {d_adv:+.4f} | {pct(d_adv)} |",
                f"| **合计 = G5 − G1** | **{total:+.4f}** | 100% |"]
        body = head + "\n".join(rows) + "\n"
        body += ("\n> 阶梯分解有顺序依赖，仅用于叙述增长曲线；结论以 Shapley 为准"
                 "（见 `reports/shapley.json`）。\n")
        return body


    def bootstrap_table() -> str:
        rows = load(REPORTS / "bootstrap.json", [])
        if not rows:
            return "_（bootstrap.json 未生成）_\n"
        head = ("| 比较 | Δ | 95% CI | p | Holm 阈值 | 显著 |\n| --- | --- | --- | --- | --- | --- |\n")
        out = []
        for r in rows:
            out.append(f"| {r['a']} − {r['b']} | {r['delta']:+.4f} | "
                       f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}] | {r['p_value']:.4f} | "
                       f"{r.get('holm_threshold', 0):.4f} | "
                       f"{'是' if r.get('significant') else '否'} |")
        return head + "\n".join(out) + "\n"


    def main(args):
        metrics = load(REPORTS / "metrics.json", {})
        TABLES.mkdir(parents=True, exist_ok=True)
        parts = {
            "main": "# 主表\n\n" + main_table(metrics),
            "ablation": "# 消融\n\n" + ablation_table(metrics),
            "routing": "# 动态路由 Pareto\n\n" + routing_table(),
            "attribution": "# 因子归因（阶梯分解）\n\n" + attribution_table(metrics),
            "bootstrap": "# 显著性检验\n\n" + bootstrap_table(),
        }
        for name, text in parts.items():
            (TABLES / f"{name}.md").write_text(text, encoding="utf-8")
        (TABLES / "all.md").write_text(
            "\n\n".join(parts[k] for k in ("main", "ablation", "routing", "attribution", "bootstrap")),
            encoding="utf-8")
        print(f"[ok] {len(parts)} tables -> {TABLES.relative_to(ROOT).as_posix()}")

    main(args)

run_stage(None, 'Cell E5: tables',
          fn=lambda: stage_tables(SimpleNamespace(**{})),
          outputs=(), always=True)


=== 14:31:49  Cell E5: tables ===


[ok] 5 tables -> reports/tables
[done] Cell E5: tables in 0.0 min (14:31:49)


True

Leave-one-out ablation inside the G5 pipeline (test set). Removing a module shows what it contributed while the rest stayed fixed.

| Variant | MTNT | EMEA | Macro | Delta vs full |
| --- | --- | --- | --- | --- |
| G5 full | 0.7298 | 0.7807 | 0.7552 | -- |
| - DPO | 0.7244 | 0.7795 | 0.7520 | -0.0032 |
| - RAG | 0.7338 | 0.7881 | 0.7610 | +0.0058 |
| - RAG (random ctx) | 0.7300 | 0.7776 | 0.7538 | -0.0014 |
| - glossary | 0.7299 | 0.7876 | 0.7588 | +0.0036 |
| - agent review | 0.7588 | 0.8018 | 0.7803 | +0.0251 |

The leave-one-out table is the most informative result in this report. Removing the agentic
critic/refiner loop gains 0.024: two extra generation passes over a 1.5B model rewrite
translations in ways that drift from the reference. The loop does not fail loudly. It produces
fluent French that is less faithful, which is exactly the failure mode a small model exhibits
when asked to critique itself.

Retrieval and glossary are smaller and negative in the same direction. Only DPO is positive, and
by an amount consistent with its isolated measurement.

**Data augmentation.** As a follow-up we tripled the noisy-domain training data: forward
translation of 43,406 monolingual English comments with the SFT model itself, back-translation
of 25,537 French comments with Opus-MT, and 5,252 human chat pairs from WMT24. The result is
0.7391 / 0.7645, a loss of 0.030 and 0.063 against the plain SFT model. Three causes are
visible in the diagnostics: output length rises by 0.05 (self-distillation amplifies the
student's own verbosity), the EMEA share of the mixture falls from 44% to 20% (terminology hit
rate drops 13 points on medical text), and real examples are seen once instead of twice. The
experiment is therefore a statement about mixing ratios under a fixed one-epoch budget, not
about data quantity.

## 7. Attribution

Additive decomposition of the ladder. The four terms sum to the total by construction. The split is order-dependent, so we read it as a narrative device and rely on the leave-one-out table for conclusions.

| Factor | Delta macro | Share of total |
| --- | --- | --- |
| $\Delta_{SFT}=G3-G1$ | +0.0510 | +630 |
| $\Delta_{RAG}=G2-G1$ | +0.0162 | +200 |
| $\Delta_{syn}=G4-G3-\Delta_{RAG}$ | -0.0316 | -390 |
| $\Delta_{adv}=G5-G4$ | -0.0275 | -340 |
| Total $=G5-G1$ | +0.0081 | 100 |

The additive split is reported because the proposal asked for factor contribution rates, but it
is order-dependent: assigning the retrieval gain before or after fine-tuning changes every
number. We therefore use the leave-one-out table for conclusions and keep this one as a
narrative summary. Exact Shapley values over the four G5 modules were considered and rejected:
with strong interactions (DPO is +0.003 alone and +0.003 inside the pipeline, while retrieval is
+0.007 alone and -0.006 inside), the Shapley split would be arithmetically clean and
interpretively empty.

## 8. What a larger model would buy

### Cell X1: ceiling arm

Rebuilds the held-out subsets and re-scores the stored teacher translations. The teacher
itself was an external API model, so its output is shipped with the repository rather
than regenerated here. Everything downstream of it is reproduced.

In [ ]:
# 上限实验：抽子集 -> 用存好的教师译文打分。教师译文来自外部 API 模型，
# 不重跑（runs/_ceiling_teacher*/pass1.jsonl 随仓库保存）；这里重做子集与打分。
if RUN["ceiling"]:
    for _dom, _n in (("mtnt", 300), ("emea", 150)):
        run_stage("ceiling", f"Cell X1: ceiling subset · {_dom}",
                  fn=lambda d=_dom, n=_n: _stream([sys.executable, "-u",
                                                   "src/eval/make_ceiling_set.py",
                                                   "--domain", d, "--n", str(n)]))
        run_stage("ceiling", f"Cell X1: ceiling scores · {_dom}",
                  fn=None, comet=("src/eval/score_ceiling.py",
                                  {"domain": _dom, "n": _n}))
else:
    print("[skip] Cell X1: RUN['ceiling'] is False", flush=True)

[skip] Cell X1: RUN['ceiling'] is False


Ceiling check on held-out subsets translated by a much larger model (GPT-6 Luna, zero-shot, no task tuning). All systems are scored on the same sentences with the same references.

| Subset | Teacher | G1 | G3 | G5* | Teacher$-$G3 |
| --- | --- | --- | --- | --- | --- |
| MTNT (300) | 0.8066 | 0.7268 | 0.7720 | 0.7747 | +0.0346 |
| EMEA (150) | 0.8223 | 0.7383 | 0.8127 | 0.8105 | +0.0096 |

This is the experiment that bounds the project. A model roughly ten times larger, translated
zero-shot and scored on the same sentences with the same references, reaches 0.8223 on the EMEA
subset against 0.8127 for our fine-tuned 1.5B model: a gap of 0.010. On MTNT the gap is 0.035,
and part of it is reference noise rather than translation quality, since MTNT references are
themselves paraphrases of noisy text.

Higher absolute scores are not available on this data by scaling the student, and the negative
results in Section 6 are not an artefact of an under-powered model: they are what happens when
the metric is close to its ceiling for the task.

## 9. Cost and reproducibility

Table 7 gives the resource envelope. The whole study fits in about 10 GPU hours on one card, of
which the two SFT runs (baseline and augmented) take 4.5.

Six defects were found and fixed during the runs, and they are worth recording because each
silently produced plausible but wrong numbers: a retrieval cache read from the wrong directory
(the heavy path ran without retrieved examples), a DPO adapter loaded onto the raw base model
instead of the merged SFT weights (DPO appeared to collapse to zero-shot), an inline context
format measured before the turn-based one, an n-gram extractor that glued words across slashes,
a metric that loaded only one of the two glossaries, and a chat-corpus parser that mixed both
translation directions. The first two changed conclusions. The third and fourth changed
component scores by an order of magnitude more than the effects being measured.

## 10. Limitations

The clean corpus is templated (2,251 unique sentences behind 45,000 rows) and its test split
leaks, so it contributes no evidence. MTNT test has 1,020 sentences, which sets the resolution
floor at 0.012; differences of 0.005 in this report are reported but not interpreted. The
glossary covers 20% of EMEA test sentences and 4% of MTNT, so the hit-rate metric rests on a
subset. Preference pairs were mined with COMET-DA and the headline metric is COMET-DA, a
circularity we mitigate with BLEU, chrF++ and the term metric but cannot remove. Single seed per
configuration. The bootstrap intervals cover sampling noise over sentences, not over training
runs.

## 11. Conclusion

On a single 16 GB card, with two hard registers and roughly 10 GPU hours, LoRA fine-tuning is
the only intervention that moves quality: +0.051 macro COMET-DA, reaching 98.8% of a model an
order of magnitude larger on medical text. Preference optimisation is real but below the
measurement resolution. Retrieval, glossary constraints, agentic self-review and dynamic routing
are neutral-to-harmful once the model is tuned, and the ablations localise the damage: context
injection costs 0.015, self-review 0.024, and neither is recovered by better retrieval pools or
domain gating. The practical recommendation for small-model translation pipelines is to spend
the budget on supervised data hygiene and stopping criteria, not on orchestration.

## Appendix: result tables

Printed from the stored per-sentence scores without retraining.

### Main results

In [ ]:
import json
D = json.loads((PROJ / "reports/report_data.json").read_text(encoding="utf-8"))
hdr = f"{'system':28s} {'MTNT':>7s} {'EMEA':>7s} {'macro':>7s} {'BLEU':>6s} {'chrF++':>7s} {'term%':>6s}"
print(hdr)
for r in D["main"]:
    print(f"{r['label']:28s} {r['mtnt']:7.4f} {r['emea']:7.4f} {r['macro']:7.4f} "
          f"{r['bleu_emea']:6.1f} {r['chrf_emea']:7.1f} {100*r['term_emea']:6.1f}")

system                          MTNT    EMEA   macro   BLEU  chrF++  term%
G1 Zero-shot                  0.7292  0.7651  0.7471   22.9    48.9   35.1
G2 Zero-shot + RAG            0.7264  0.8002  0.7633   28.8    54.0   56.7
G3 LoRA SFT                   0.7691  0.8271  0.7981   39.6    61.9   70.2
G5* SFT + DPO                 0.7736  0.8294  0.8015   40.8    62.6   72.5
G4 SFT + RAG                  0.7544  0.8110  0.7827   35.6    58.4   66.8
G4 (inline context)           0.6688  0.7780  0.7234   32.8    55.8   65.1
G4 (domain-gated)             0.7541  0.8106  0.7823   35.7    58.4   66.8
SFT + augmented data          0.7391  0.7645  0.7518   32.3    55.2   56.9
G5 full pipeline              0.7298  0.7807  0.7552   33.8    58.4   75.8


### Significance

In [ ]:
for b in D["bootstrap"]:
    print(f"{b['a']:16s} - {b['b']:16s} {b['delta']:+.4f} "
          f"[{b['ci_lo']:+.4f}, {b['ci_hi']:+.4f}] p={b['p_value']:.4f} "
          f"{'sig' if b.get('significant') else 'n.s.'}")

g5_full          - g3_sft           -0.0440 [-0.0478, -0.0401] p=0.0000 sig
g5_full          - g5_no_dpo        +0.0027 [+0.0001, +0.0052] p=0.0412 n.s.
g5_full          - g5_no_rag        -0.0062 [-0.0100, -0.0026] p=0.0012 sig
g5_full          - g5_no_rag_len    +0.0020 [-0.0020, +0.0061] p=0.3328 n.s.
g5_full          - g5_no_con        -0.0046 [-0.0067, -0.0026] p=0.0000 sig
g5_full          - g5_no_agent      -0.0238 [-0.0267, -0.0209] p=0.0000 sig
g5_dpo_direct    - g3_sft           +0.0031 [+0.0015, +0.0046] p=0.0000 sig
g3_sft_aug       - g3_sft           -0.0516 [-0.0560, -0.0472] p=0.0000 sig


### Routing sweep

In [ ]:
rt = D["routing"]
print("light", rt["reference"]["always_light"], "heavy", rt["reference"]["always_heavy"])
for pt in rt["points"]:
    tag = f"tau={pt['tau']}" if pt["tau"] is not None else pt["label"]
    print(f"{tag:16s} macro={pt['macro']:.4f} lat={pt['latency_ms']:6.0f}ms "
          f"heavy={100*pt['heavy_ratio']:.0f}%")
print("chosen", rt["chosen_tau"])

light 0.7974 heavy 0.7553
always_light     macro=0.7974 lat=   118ms heavy=0%
always_heavy     macro=0.7553 lat=   492ms heavy=100%
tau=0.5          macro=0.7974 lat=   128ms heavy=3%
tau=0.55         macro=0.7974 lat=   133ms heavy=4%
tau=0.6          macro=0.7979 lat=   141ms heavy=6%
tau=0.65         macro=0.7975 lat=   154ms heavy=10%
tau=0.7          macro=0.7956 lat=   171ms heavy=14%
tau=0.75         macro=0.7936 lat=   196ms heavy=21%
tau=0.8          macro=0.7891 lat=   233ms heavy=31%
tau=0.85         macro=0.7792 lat=   302ms heavy=49%
chosen {'tau': 0.6, 'macro': 0.7979, 'by_domain': {'mtnt': 0.7691, 'emea': 0.8267}, 'latency_ms': 141.4, 'heavy_ratio': 0.063}


### Ceiling

In [ ]:
for dom, c in D["ceiling_same_subset"].items():
    print(dom, c)

mtnt {'teacher': 0.8066, 'g1_zs': 0.7268, 'g2_rag': 0.7254, 'g3_sft': 0.772, 'g5_dpo_direct': 0.7747}
emea {'teacher': 0.8223, 'g1_zs': 0.7383, 'g2_rag': 0.7862, 'g3_sft': 0.8127, 'g5_dpo_direct': 0.8105}


## Run summary

What executed in this run, how long each stage took, and the headline metric of every
system in Table 3 read back from `reports/metrics.json`.

In [ ]:
summary()


stage                             status      min
-------------------------------------------------
Cell D1: build the three corpora  skipped     0.0
Cell D2: retrieval index          skipped     0.0
Cell D3: terminology alignment    skipped     0.0
Cell D4: finalise the glossary    skipped     0.0
Cell T1: LoRA SFT                 skipped     0.0
Cell T2: mine preference pairs    skipped     0.0
Cell T3: score the candidates     skipped     0.0
Cell T4: DPO                      skipped     0.0
Cell I1: batch inference          skipped     0.0
Cell I2: G5 pipeline              skipped     0.0
Cell E1: COMET scoring            skipped     0.0
Cell E2: metrics                  ok          0.4
Cell E3: paired bootstrap         ok          1.3
Cell E4: routing sweep            ok          0.0
Cell E5: tables                   ok          0.0

4 stage(s) executed, wall clock 1.9 min

run                       MTNT    EMEA   macro
g1_zs                   0.7292  0.7651  0.7471
g2_rag       

## References

1. A. Vaswani et al., Attention Is All You Need, NeurIPS 2017.
2. E. J. Hu et al., LoRA: Low-Rank Adaptation of Large Language Models, ICLR 2022.
3. Qwen Team, Qwen2.5 Technical Report, 2024.
4. P. Michel and G. Neubig, MTNT: A Testbed for Machine Translation of Noisy Text, EMNLP 2018.
5. J. Tiedemann, Parallel Data, Tools and Interfaces in OPUS, LREC 2012.
6. R. Rafailov et al., Direct Preference Optimization, NeurIPS 2023.
7. R. Rei et al., CometKiwi: IST-Unbabel 2022 Submission for the Quality Estimation Shared Task, WMT 2022.
8. R. Rei et al., COMET: A Neural Framework for MT Evaluation, EMNLP 2020.
9. P. Koehn, Statistical Significance Tests for Machine Translation Evaluation, EMNLP 2004.